In [ ]:

import numpy as np
import pandas as pd
import MDAnalysis as mda
import matplotlib.pyplot as plt
#import plotly.express as px
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pickle
import parmed
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


import matplotlib.ticker as ticker
from matplotlib.colors import TwoSlopeNorm
import io


In [ ]:
plt.style.use('./paper_style.mpl')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


df = pd.read_csv('Figures/descriptor.csv')
df_2 =  pd.read_csv('Figures/observables.csv')

df_2 = df_2[['system','cluster_size','sasa_per_atom','protein t']]
# ─────────────────────────────────────────────────────────────
# IMPORTANT:
# Your df has 'system' as a COLUMN
# not as index
# ─────────────────────────────────────────────────────────────

# set system as index
df = df.set_index('system')
df_2 = df_2.set_index('system')

# ─────────────────────────────────────────────────────────────
# DEFINE GROUPS
# ─────────────────────────────────────────────────────────────
groups = {
    '3-mer'  : ['3E','3C','3A','3P','3N'],
    '9-mer'  : ['9E','9C','9A','9P','9N'],
    'Spacer' : ['X2','X2S','X3','X3S']
}

# ─────────────────────────────────────────────────────────────
# KEEP ONLY NUMERIC COLUMNS
# ─────────────────────────────────────────────────────────────
df_num  = df.select_dtypes(include=np.number)
df2_num = df_2.select_dtypes(include=np.number)

# ─────────────────────────────────────────────────────────────
# FUNCTION TO BUILD CROSS-CORRELATION MATRIX
# rows = descriptors
# cols = observables
# ─────────────────────────────────────────────────────────────
def build_cross_corr(dfA, dfB):

    corr = pd.DataFrame(
        index=dfA.columns,
        columns=dfB.columns,
        dtype=float
    )

    for c1 in dfA.columns:
        for c2 in dfB.columns:

            val = dfA[c1].corr(dfB[c2])

            # replace NaN with 0
            if pd.isna(val):
                val = 0.0

            corr.loc[c1, c2] = val

    return corr

# ─────────────────────────────────────────────────────────────
# CALCULATE GROUP-WISE CORRELATIONS
# ─────────────────────────────────────────────────────────────
corr_results = {}
corr_matrices = []

for group_name, systems in groups.items():

    # select same systems in both dfs
    sub_df  = df_num.loc[systems]
    sub_df2 = df2_num.loc[systems]

    # build correlation matrix
    corr = build_cross_corr(sub_df, sub_df2)

    corr_results[group_name] = corr
    corr_matrices.append(corr)

    print(f"\n{'='*60}")
    print(group_name)
    print('='*60)

    print(corr.round(3))

# ─────────────────────────────────────────────────────────────
# AVERAGE CORRELATION MATRIX
# ─────────────────────────────────────────────────────────────
avg_corr = sum(corr_matrices) / len(corr_matrices)

print(f"\n{'='*60}")
print("AVERAGE CORRELATION MATRIX")
print('='*60)

print(avg_corr.round(3))

# ─────────────────────────────────────────────────────────────
# PLOT AVERAGE HEATMAP
# ─────────────────────────────────────────────────────────────
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

fig, ax = plt.subplots(figsize=(7,5.5))

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im = ax.imshow(
    avg_corr.values,
    cmap='RdBu_r',
    norm=norm,
    aspect='auto'
)

# grid
n_rows, n_cols = avg_corr.shape

ax.set_xticks(np.arange(n_cols + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n_rows + 1) - 0.5, minor=True)

ax.grid(which='minor', linewidth=2)

ax.tick_params(which='minor', bottom=False, left=False)

# ticks
ax.set_xticks(np.arange(n_cols))
ax.set_yticks(np.arange(n_rows))

ax.set_xticklabels(
    avg_corr.columns,
    rotation=35,
    ha='right',
    fontsize=10
)

ax.set_yticklabels(
    avg_corr.index,
    fontsize=10
)

# annotate cells
for i in range(n_rows):
    for j in range(n_cols):

        val = avg_corr.values[i, j]

        txt_color = (
            'white' if abs(val) > 0.55 else 'black'
        )

        ax.text(
            j,
            i,
            f"{val:.2f}",
            ha='center',
            va='center',
            fontsize=8,
            color=txt_color
        )

# colorbar
cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.02
)

cbar.set_label("Pearson r", fontsize=10)

# title
ax.set_title(
    "Average Correlation Across 3-mer, 9-mer and Spacer Systems",
    fontsize=13,
    fontweight='bold',
    pad=12
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

df = pd.read_csv('Figures/descriptor.csv')
df_2 =  pd.read_csv('Figures/observables.csv')

df_2 = df_2[['system','cluster_size','sasa_per_atom','tau_all']]

# ─────────────────────────────────────────────────────────────
# set system as index
# ─────────────────────────────────────────────────────────────
df    = df.set_index('system')
df_2  = df_2.set_index('system')

# ─────────────────────────────────────────────────────────────
# DEFINE GROUPS
# ─────────────────────────────────────────────────────────────
groups = {
    '3-mer'  : ['3E','3C','3A','3P','3N'],
    '9-mer'  : ['9E','9C','9A','9P','9N'],
    'Spacer' : ['X2','X2S','X3','X3S'],
}

# ─────────────────────────────────────────────────────────────
# NUMERIC COLUMNS ONLY
# ─────────────────────────────────────────────────────────────
df_num  = df.select_dtypes(include=np.number)
df2_num = df_2.select_dtypes(include=np.number)

# ─────────────────────────────────────────────────────────────
# CROSS-CORRELATION BUILDER
# ─────────────────────────────────────────────────────────────
def build_cross_corr(dfA, dfB):
    corr = pd.DataFrame(index=dfA.columns, columns=dfB.columns, dtype=float)
    for c1 in dfA.columns:
        for c2 in dfB.columns:
            val = dfA[c1].corr(dfB[c2])
            corr.loc[c1, c2] = 0.0 if pd.isna(val) else val
    return corr

# ─────────────────────────────────────────────────────────────
# COMPUTE PER-GROUP CORRELATIONS
# ─────────────────────────────────────────────────────────────
corr_results = {}
for group_name, systems in groups.items():
    corr_results[group_name] = build_cross_corr(
        df_num.loc[systems], df2_num.loc[systems]
    )

# ─────────────────────────────────────────────────────────────
# PLOT: 3 panels side by side + 1 shared colorbar
# ─────────────────────────────────────────────────────────────
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"]  = ["Times New Roman"]

norm  = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
cmap  = "RdBu_r"

n_groups = len(groups)
group_names = list(groups.keys())

# figure: 3 heatmap columns + 1 narrow colorbar column
fig = plt.figure(figsize=(18, 6.5))
gs  = gridspec.GridSpec(
    1, n_groups + 1,
    width_ratios=[1, 1, 1, 0.06],
    wspace=0.08
)

axes = [fig.add_subplot(gs[0, i]) for i in range(n_groups)]
cax  =  fig.add_subplot(gs[0, n_groups])

for ax_idx, (gname, ax) in enumerate(zip(group_names, axes)):

    corr  = corr_results[gname]
    n_rows, n_cols = corr.shape

    im = ax.imshow(corr.values, cmap=cmap, norm=norm, aspect="auto")

    # minor grid
    ax.set_xticks(np.arange(n_cols + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(n_rows + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # x ticks (all panels)
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(corr.columns, rotation=40, ha="right", fontsize=9)

    # y ticks: only leftmost panel
    ax.set_yticks(np.arange(n_rows))
    if ax_idx == 0:
        ax.set_yticklabels(corr.index, fontsize=9.5)
    else:
        ax.set_yticklabels([])

    # annotate cells
    for i in range(n_rows):
        for j in range(n_cols):
            val = corr.values[i, j]
            txt_color = "white" if abs(val) > 0.60 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=7.5, color=txt_color)

    # group label as title; include n so reader knows df is small
    n = len(groups[gname])
    ax.set_title(f"{gname}  (n={n})", fontsize=12, fontweight="bold", pad=8)

    # subtle border to visually separate panels
    for spine in ax.spines.values():
        spine.set_edgecolor("#aaaaaa")
        spine.set_linewidth(1.2)

# shared colorbar
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Pearson  r", fontsize=10, labelpad=8)
cbar.ax.tick_params(labelsize=9)

fig.suptitle(
    "Descriptor × Observable Correlations by System Group",
    fontsize=14, fontweight="bold", y=1.02
)


plt.show()
print("Saved → correlation_panel.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

df = pd.read_csv('Figures/observables.csv')
df_2 = pd.read_csv('Figures/descriptor.csv')

df  = df[['system','cluster_size','percolation','sasa_per_atom','protein t']]
df_2  = df_2[['system','FCR','f_aromatic','f_sticker','hydropathy','sc_volume',
             'kappa_xg','sticker_spread','chain']]
# ─────────────────────────────────────────────────────────────
# set system as index
# ─────────────────────────────────────────────────────────────
df    = df.set_index('system')
df_2  = df_2.set_index('system')

# ─────────────────────────────────────────────────────────────
# DEFINE GROUPS
# ─────────────────────────────────────────────────────────────
groups = {
    '3-mer'  : ['3E','3C','3A','3P','3N'],
    '9-mer'  : ['9E','9C','9A','9P','9N'],
    'Spacer' : ['X2','X2S','X3','X3S'],
}

# ─────────────────────────────────────────────────────────────
# NUMERIC COLUMNS ONLY
# ─────────────────────────────────────────────────────────────
df_num  = df.select_dtypes(include=np.number)
df2_num = df_2.select_dtypes(include=np.number)

# ─────────────────────────────────────────────────────────────
# CROSS-CORRELATION BUILDER — NaN stays NaN (constant column)
# ─────────────────────────────────────────────────────────────
def build_cross_corr(dfA, dfB):
    corr = pd.DataFrame(index=dfA.columns, columns=dfB.columns, dtype=float)
    for c1 in dfA.columns:
        for c2 in dfB.columns:
            corr.loc[c1, c2] = dfA[c1].corr(dfB[c2])  # NaN preserved
    return corr

# ─────────────────────────────────────────────────────────────
# COMPUTE PER-GROUP CORRELATIONS
# ─────────────────────────────────────────────────────────────
corr_results = {}
corr_arrays  = []   # list of 2-D float arrays (NaN where undefined)

for group_name, systems in groups.items():
    corr = build_cross_corr(df_num.loc[systems], df2_num.loc[systems])
    corr_results[group_name] = corr
    corr_arrays.append(corr.values.astype(float))

# ─────────────────────────────────────────────────────────────
# GLOBAL CORRELATION (all 14 systems) — fallback for descriptors
# that are constant within every group (e.g. 'chain' is always
# 3 inside 3-mer, always 9 inside 9-mer/Spacer → NaN group-wise,
# but varies meaningfully across the full dataset).
# ─────────────────────────────────────────────────────────────
global_corr = build_cross_corr(df_num, df2_num)

# ─────────────────────────────────────────────────────────────
# SMART AVERAGE:
#   - cells valid in ≥1 group  → np.nanmean over valid groups
#   - cells NaN in ALL groups  → use global correlation instead
#
# n_valid tracks how many groups contributed:
#   n_valid == 0  means global fallback was used (annotated †)
#   0 < n_valid < n_groups means partial average (annotated *)
# ─────────────────────────────────────────────────────────────
stack    = np.array(corr_arrays)                    # (n_groups, n_rows, n_cols)
avg_vals = np.nanmean(stack, axis=0)                # NaN only if ALL groups NaN
n_valid  = np.sum(~np.isnan(stack), axis=0)         # groups that contributed

# where all groups were NaN → fill from global correlation
all_nan_mask = (n_valid == 0)
avg_vals[all_nan_mask] = global_corr.values[all_nan_mask]

avg_corr = pd.DataFrame(
    avg_vals,
    index=corr_results[list(groups.keys())[0]].index,
    columns=corr_results[list(groups.keys())[0]].columns,
)

print("\nAverage correlation matrix (nanmean):")
print(avg_corr.round(3))
print("\nNumber of groups contributing per cell:")
print(pd.DataFrame(n_valid, index=avg_corr.index, columns=avg_corr.columns))

# ─────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"]  = ["Times New Roman"]

fig, ax = plt.subplots(figsize=(5,2.2))

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
im   = ax.imshow(avg_corr.values, cmap="RdBu_r", norm=norm, aspect="auto")

n_rows, n_cols = avg_corr.shape

# grid
ax.set_xticks(np.arange(n_cols + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n_rows + 1) - 0.5, minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

# axis labels
ax.set_xticks(np.arange(n_cols))
ax.set_yticks(np.arange(n_rows))

x_label = ['FCR', 'f_aromatic', 'f_sticker', 'hydropathy', 
           'sc_volume', 'kappa_xg', 'sticker_spread', 'chain_length']
y_label = ['cluster_size', 'percolation', 'sasa_per_atom', 'protein t']


x_label = [
    r'$FCR$',
    r'$f_{\mathrm{aro}}$',
    r'$f_{\mathrm{stic}}$',
    r'$\lambda$' ,                 # hydropathy (often denoted by λ)
    r'$V_{\mathrm{sc}}$',
    r'$\kappa_{\mathrm{xg}}$',
    r'$S_{\mathrm{sticker}}$',
    r'$N_{\mathrm{chain}}$'
]

y_label = [
    'cluster_size',     # or Cluster Size
    'percolation',
    'sasa_per_atom',
    r'$\tau_{\mathrm{protein}}$'
]

# ax.set_xticklabels(avg_corr.columns, rotation=40, ha="right", fontsize=10)
# ax.set_yticklabels(avg_corr.index, fontsize=10)

ax.set_xticklabels(x_label, rotation=40, ha="right", fontsize=10)
ax.set_yticklabels(y_label, fontsize=10)

# annotate cells
# † = global correlation used (descriptor constant in all groups)
# * = partial average (descriptor constant in some groups only)
n_groups = len(groups)
for i in range(n_rows):
    for j in range(n_cols):
        val = avg_corr.values[i, j]
        n_g = n_valid[i, j]
        if n_g == 0:
            suffix = "†"           # global fallback
        elif n_g < n_groups:
            suffix = "*"           # partial average
        else:
            suffix = ""
        tcolor = "white" if abs(val) > 0.60 else "black"
        ax.text(j, i, f"{val:.2f}{suffix}", ha="center", va="center",
                fontsize=8, color=tcolor)

# colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Pearson  r", fontsize=10)
cbar.ax.tick_params(labelsize=9)

# ax.set_title(
#     "Average Descriptor × Observable Correlation\n"
#     "(† = global correlation across all 14 systems;  * = partial group average)",
#     fontsize=11, fontweight="bold", pad=10
# )

# for spine in ax.spines.values():
#     spine.set_edgecolor("#aaaaaa")

plt.tight_layout()
plt.savefig('Figures/correlation.png',dpi=500)
plt.show()
print("Saved → correlation_heatmap_avg.png")


In [ ]:
avg_corr.index

In [ ]:
x_label = ['FCR', 'f_aromatic', 'f_sticker', 'hydropathy', 'sc_volume', 'kappa_xg',
       'sticker_spread', 'chain_length']
y_label = ['cluster_size', 'percolation', 'sasa_per_atom', 'protein t']


In [ ]:
## Figure_2 

In [ ]:
## Size data

total_size_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/{i}/{j}_size.csv', 
                              index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = temp_df['prot-size'].values
        total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    #num_stickers = 1
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/close_spread/{i}_size.csv',
                          index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = temp_df['prot-size'].values / num_stickers
    total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)


total_size_df

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})

## SASA data

total_sasa_df = pd.DataFrame()

# -------------------------------
# 3res and 9res systems
# -------------------------------
systems = ['equal','charged','aromatic','polar','nonpolar']

df_3res = pd.read_csv('data/sasa_persidue_3res.csv')
df_9res = pd.read_csv('data/sasa_persidue_9res.csv')

for j in systems:
    temp_df = pd.DataFrame()
    temp_df[f'3{j[0].upper()}'] = df_3res[j].values
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

for j in systems:
    temp_df = pd.DataFrame()
    temp_df[f'9{j[0].upper()}'] = df_9res[j].values
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

# -------------------------------
# Spacer systems (9res_spacers)
# -------------------------------
df_9res_sp = pd.read_csv('data/sasa_persidue_9res_spacers.csv')

spacer_map = {
    '2_close':  ('X2', 2),
    '2_spread': ('X2S', 2),
    '3_close':  ('X3', 3),
    '3_spread': ('X3S', 3)
}

for key, (label, n_stickers) in spacer_map.items():
    temp_df = pd.DataFrame()
    
    # assuming column names in df_9res_sp match keys
    temp_df[label] = df_9res_sp[key].values
    
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

# -------------------------------
# Final dataframe
total_sasa_df.head()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------------------------------------------------------
# Config
# -------------------------------------------------------
ORDER = ["3E", "3C", "3A", "3P", "3N",
         "9E", "9C", "9A", "9P", "9N",
         "X2", "X2S", "X3", "X3S"]

CLASS_COLOR = {
    **{s: "#1f77b4" for s in ["3E", "3C", "3A", "3P", "3N"]},
    **{s: "#2ca02c" for s in ["9E", "9C", "9A", "9P", "9N"]},
    **{s: "#d62728" for s in ["X2", "X2S", "X3", "X3S"]},
}
#HL = {"X3S": "#d62728", "X3": "#ff7f0e"}
HL = {}
def col(s):
    return HL.get(s, CLASS_COLOR[s])

NUDGE = {
    "X2":  (-0.025,  0.0014),
    "X2S": (+0.028, +0.0000),
    "9E":  (+0.035, -0.0020),
    "9C":  (+0.030,  0.0010),
    "9A":  (-0.030,  0.0014),
    "9P":  (-0.02,  -0.000),
    "9N":  (-0.025, -0.0010),
    "X3":  (-0.02, -0.0010),
    "X3S": (+0.024,  0.0010),
    "3C":  (+0.020,  0.0006),
    "3A":  (+0.020, -0.0006),
    "3P":  (+0.030,  -0.0006),
    "3E":  (-0.020, -0.0006),
    "3N":  (-0.020, -0.0006),
}

# -------------------------------------------------------
# Means from your DataFrames
# -------------------------------------------------------
size_mean = total_size_df.mean()
sasa_mean = total_sasa_df.mean()

# -------------------------------------------------------
# Scatter plot
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(3.6, 2.2))

xs = size_mean.values
ys = sasa_mean.values
ax.set_xlim(xs.min() - 0.05, xs.max() + 0.04)
ax.set_ylim(ys.min() - 0.004, ys.max() + 0.005)

for system in size_mean.index:
    x = size_mean[system]
    y = sasa_mean[system]
    is_hl = system in HL
    dx, dy = NUDGE.get(system, (0.0, 0.0))
    xx, yy = x + dx, y + dy

    # Dot at true data position
    ax.scatter(x, y,
               marker="o",
               s=22 if is_hl else 12,
               color=col(system),
               zorder=2, alpha=0.85)

    # Leader line if nudged
    if dx != 0 or dy != 0:
        ax.plot([x, xx], [y, yy], color="grey", lw=0.4, zorder=1)

    # Text label near point
    ax.text(xx, yy, system,
            ha="center", va="center",
            #fontsize=9,
            fontweight="bold",
            color=col(system),
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=0.5),
            zorder=3)

# -------------------------------------------------------
# Labels & legend
# -------------------------------------------------------
ax.set_xlabel("largest-cluster fraction (peptides in dense phase)", fontsize=8)
ax.set_ylabel(r"SASA per atom  (nm²)", fontsize=8)
ax.grid(alpha=0.3)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

leg = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#1f77b4",
           markeredgecolor="#1f77b4", markersize=6, label="3-residue"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ca02c",
           markeredgecolor="#2ca02c", markersize=6, label="9-residue"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d62728",
           markeredgecolor="#d62728", markersize=6, label="patterned"),
#     Line2D([0], [0], marker="o", color="w", markerfacecolor="#ff7f0e",
#            markeredgecolor="#ff7f0e", markersize=9, label="X3 / X3S"),
]
ax.legend(handles=leg, loc="lower left", fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig("Figures/cluster_size_sasa.png", dpi=400)
plt.show()

In [ ]:
## Compositional analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
#import plotly.express as px
import numpy as np
import MDAnalysis as mda
import matplotlib.patches as mpatches
import matplotlib.patheffects as path_effects



equal = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/equal/comp.csv', index_col=0)
charged = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/charged/comp.csv', index_col=0)
aromatic = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/aromatic/comp.csv', index_col=0)
polar = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/polar/comp.csv', index_col=0)
nonpolar = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/nonpolar/comp.csv', index_col=0)

for i in [equal,charged,aromatic,polar,nonpolar]:
     i = i.sort_index()

res3 = pd.DataFrame(index=equal.index)

res3['equal'] = equal['frequency']
res3['charged'] = charged['frequency']
res3['aromatic'] = aromatic['frequency']
res3['polar'] = polar['frequency']
res3['nonpolar'] = nonpolar['frequency']

res3 = res3.sort_index()

equal = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/old_equal/comp.csv', index_col=0)
charged = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/charged/comp.csv', index_col=0)
aromatic = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/aromatic/comp.csv', index_col=0)
polar = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/polar/comp.csv', index_col=0)
nonpolar = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/nonpolar/comp.csv', index_col=0)

for i in [equal,charged,aromatic,polar,nonpolar]:
     i = i.sort_index()

res9 = pd.DataFrame(index=equal.index)

res9['equal'] = equal['frequency']
res9['charged'] = charged['frequency']
res9['aromatic'] = aromatic['frequency']
res9['polar'] = polar['frequency']
res9['nonpolar'] = nonpolar['frequency']

res9 = res9.sort_index()


In [ ]:
tot_df = res3

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# -----------------------------
# Residue groups
# -----------------------------
aromatic = ['PHE', 'TYR', 'TRP', 'HIS']
polar    = ['CYS', 'THR', 'GLN', 'ASN', 'SER']
positive = ['ARG', 'LYS']
negative = ['GLU', 'ASP']
nonpolar = ['VAL', 'ILE', 'ALA', 'LEU', 'PRO', 'GLY', 'MET']

ordered_residues = aromatic + polar + positive + negative + nonpolar
systems = ['equal', 'charged', 'aromatic', 'polar', 'nonpolar']
labels  = ['3E', '3C', '3A', '3P', '3N']

# -----------------------------
# Build 2D matrix  (systems × residues)
# -----------------------------
# tot_df rows = residues, columns = systems
matrix = tot_df.loc[ordered_residues, systems].values.T   # shape: (n_systems, n_residues)

# -----------------------------
# Background colour per residue class
# -----------------------------
class_colors = {
    'Aromatic' : '#e3ba14',
    'Polar'    : '#7aaa3d',
    'Positive' : '#b30039',
    'Negative' : '#0c7ace',
    'Nonpolar' : '#ac5cad',
}
sections = [
    (aromatic,  'Aromatic'),
    (polar,     'Polar'),
    (positive,  'Positive'),
    (negative,  'Negative'),
    (nonpolar,  'Nonpolar'),
]

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(4.9, 2.2))

cmap = plt.cm.YlOrRd          # or choose any diverging/sequential cmap you like
vmin, vmax = 0.5, 1.0

im = ax.imshow(
    matrix,
    aspect='auto',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    interpolation='nearest'
)

# -----------------------------
# Background shading per residue class
# -----------------------------
x_cursor = 0
for group, class_name in sections:
    n = len(group)
    ax.axvspan(x_cursor - 0.5, x_cursor + n - 0.5,
               color=class_colors[class_name], alpha=0.08, zorder=0)
    # class label above the heatmap
    ax.text(x_cursor + (n - 1) / 2, 1, class_name,
            ha='center', va='bottom', fontsize=6.5,
            color=class_colors[class_name], fontweight='bold',
            transform=ax.get_xaxis_transform())
    x_cursor += n

# -----------------------------
# Vertical separators between classes
# -----------------------------
boundaries = np.cumsum([len(g) for g, _ in sections[:-1]])
for b in boundaries:
    ax.axvline(b - 0.5, color='black', linestyle='--', lw=0.6, alpha=0.35)

# -----------------------------
# Axes ticks & labels
# -----------------------------
ax.set_xticks(range(len(ordered_residues)))
ax.set_xticklabels(ordered_residues, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('Residue', fontsize=7)
ax.set_ylabel('System', fontsize=7)

# optional: print value in each cell
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=5, color='black' if val < 0.8 else 'white')

# -----------------------------
# Colour bar
# -----------------------------
cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.015)
cbar.set_label('Clustered Ratio', fontsize=7)
cbar.ax.tick_params(labelsize=6)
plt.minorticks_off()
plt.tight_layout()
plt.savefig('Figures/cluster_size_composition_heatmap.png', dpi=400)
plt.show()

In [ ]:
## 9_residues systems

In [ ]:
tot_df = res9

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# -----------------------------
# Residue groups
# -----------------------------
aromatic = ['PHE', 'TYR', 'TRP', 'HIS']
polar    = ['CYS', 'THR', 'GLN', 'ASN', 'SER']
positive = ['ARG', 'LYS']
negative = ['GLU', 'ASP']
nonpolar = ['VAL', 'ILE', 'ALA', 'LEU', 'PRO', 'GLY', 'MET']

ordered_residues = aromatic + polar + positive + negative + nonpolar
systems = ['equal', 'charged', 'aromatic', 'polar', 'nonpolar']
labels  = ['9E', '9C', '9A', '9P', '9N']

# -----------------------------
# Build 2D matrix  (systems × residues)
# -----------------------------
# tot_df rows = residues, columns = systems
matrix = tot_df.loc[ordered_residues, systems].values.T   # shape: (n_systems, n_residues)

# -----------------------------
# Background colour per residue class
# -----------------------------
class_colors = {
    'Aromatic' : '#e3ba14',
    'Polar'    : '#7aaa3d',
    'Positive' : '#b30039',
    'Negative' : '#0c7ace',
    'Nonpolar' : '#ac5cad',
}
sections = [
    (aromatic,  'Aromatic'),
    (polar,     'Polar'),
    (positive,  'Positive'),
    (negative,  'Negative'),
    (nonpolar,  'Nonpolar'),
]

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(4.9, 2.2))

cmap = plt.cm.YlOrRd          # or choose any diverging/sequential cmap you like
vmin, vmax = 0.5, 1.0

im = ax.imshow(
    matrix,
    aspect='auto',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    interpolation='nearest'
)

# -----------------------------
# Background shading per residue class
# -----------------------------
x_cursor = 0
for group, class_name in sections:
    n = len(group)
    ax.axvspan(x_cursor - 0.5, x_cursor + n - 0.5,
               color=class_colors[class_name], alpha=0.08, zorder=0)
    # class label above the heatmap
    ax.text(x_cursor + (n - 1) / 2, 1, class_name,
            ha='center', va='bottom', fontsize=5.5,
            color=class_colors[class_name], fontweight='bold',
            transform=ax.get_xaxis_transform())
    x_cursor += n

# -----------------------------
# Vertical separators between classes
# -----------------------------
boundaries = np.cumsum([len(g) for g, _ in sections[:-1]])
for b in boundaries:
    ax.axvline(b - 0.5, color='black', linestyle='--', lw=0.6, alpha=0.35)

# -----------------------------
# Axes ticks & labels
# -----------------------------
ax.set_xticks(range(len(ordered_residues)))
ax.set_xticklabels(ordered_residues, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('Residue', fontsize=7)
ax.set_ylabel('System', fontsize=7)

# optional: print value in each cell
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=4, color='black' if val < 0.8 else 'white')

# -----------------------------
# Colour bar
# -----------------------------
cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.015)
cbar.set_label('Clustered Ratio', fontsize=7)
cbar.ax.tick_params(labelsize=6)
plt.minorticks_off()
plt.tight_layout()
plt.savefig('Figures/cluster_size_composition_heatmap_9res.png', dpi=400)
plt.show()

In [ ]:

from fun_corr import * 
import numpy as np
import matplotlib.pyplot as plt
import pickle


In [ ]:

# ============================================================
# DATASETS
# ============================================================
datasets = [

    ('corr_results/3res_corr_results_2.pkl',
     ['equal','charged' ,'aromatic', 'polar', 'nonpolar'],
     '3res'),

    ('corr_results/9res_corr_results_2.pkl',
     ['equal','charged' ,'aromatic', 'polar', 'nonpolar'],
     '9res'),

    ('corr_results/9res_corr_results_spacers.pkl',
     ['2_close', '2_spread', '3_close', '3_spread'],
     '9res (spacers)'),
]

# ============================================================
# LABEL MAP
# ============================================================
label_map = {

    # 3res
    ('3res', 'equal')    : '3E',
    ('3res', 'aromatic') : '3A',
    ('3res', 'nonpolar') : '3N',
    ('3res', 'polar')    : '3P',
    ('3res', 'charged')  : '3C',

    # 9res
    ('9res', 'equal')    : '9E',
    ('9res', 'aromatic') : '9A',
    ('9res', 'nonpolar') : '9N',
    ('9res', 'polar')    : '9P',
    ('9res', 'charged')  : '9C',

    # spacer systems
    ('9res (spacers)', '2_close')  : 'X2',
    ('9res (spacers)', '2_spread') : 'X2S',
    ('9res (spacers)', '3_close')  : 'X3',
    ('9res (spacers)', '3_spread') : 'X3S',
}

# ============================================================
# COLORS
# ============================================================
clrs = ['black', 'red', 'blue', 'green', 'purple', 'orange']

plt.rcParams.update({
    "font.size": 14,
})

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

# ============================================================
# FIGURE
# ============================================================
fig,axes = plt.subplots(1,3,figsize=(7,2),sharey=True)

# top row = 3 correlation plots
ax1 =  axes[0]
ax2 = axes[1]
ax3 = axes[2]
top_axes = [ax1, ax2, ax3]
# bottom row = one large comparison plot

# ============================================================
# STORE ALL TAU VALUES
# ============================================================
all_tau = []
all_labels = []
all_colors = []


# ============================================================
# LOOP OVER DATASETS
# ============================================================
for col, (save_path, systems, title) in enumerate(datasets):

    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = top_axes[col]

    # ========================================================
    # CORRELATION PLOTS
    # ========================================================
    for i, label in enumerate(systems):

        r = results[label]
        r['tM']=r['tM']
        
        pretty_label = label_map[(title, label)]

        # ----------------------------------------------------
        # raw correlation points
        # ----------------------------------------------------
        ax_corr.plot(
            r['tM'][::5],
            r['corrN'][::5],
            '--',
            color=clrs[i],
            alpha=0.25,
            markersize=3
        )

        # ----------------------------------------------------
        # fit
        # ----------------------------------------------------
        try:

            popt, tau_avg = fit_exp3(r['tM'], r['corrN'])
            yfit = exp3(r['tM'], *popt)

            ax_corr.plot(
                r['tM'],
                yfit,
                '-',
                color=clrs[i],
                lw=2,
                label=f'{pretty_label}'
            )

            # store for bottom comparison
            all_tau.append(tau_avg)
            all_labels.append(pretty_label)
            all_colors.append(clrs[i])

        except RuntimeError:

            all_tau.append(np.nan)
            all_labels.append(pretty_label)
            all_colors.append(clrs[i])

    # ========================================================
    # SETTINGS
    # ========================================================
    ax_corr.set_title(title, fontsize=10)

    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")

    #ax_corr.set_ylim((0,0.3))

    ax_corr.legend(
        frameon=False,
        fontsize=8,ncol=2
    )



In [ ]:
all_tau_protein = all_tau
all_labels_protein = all_labels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- labels, in the same order as all_param ----
groups = [
    ('3res',           ['equal', 'charged', 'aromatic', 'polar', 'nonpolar']),
    ('9res',           ['equal', 'charged', 'aromatic', 'polar', 'nonpolar']),
    ('9res (spacers)', ['2_close', '2_spread', '3_close', '3_spread']),
]

P = np.array([np.ravel(x[0]) for x in all_param])   # shape (14, 5)
assert len(P) == sum(len(l) for _, l in groups)

rows, i = [], 0
for gname, labels in groups:
    for lab in labels:
        A1, A2, T1, T2, T3 = P[i]
        A = np.array([A1, A2, 1 - A1 - A2])
        T = np.array([T1, T2, T3])
        order = np.argsort(T)                 # fast -> mid -> slow
        A, T = A[order], T[order]
        tau_avg = np.sum(A * T)               # amplitude-weighted
        tau_int = np.sum(A * T**2) / tau_avg  # intensity-weighted (for reference)
        rows.append(dict(group=gname, cond=lab,
                         A_fast=A[0], A_mid=A[1], A_slow=A[2],
                         T_fast=T[0], T_mid=T[1], T_slow=T[2],
                         f_fast=A[0]*T[0]/tau_avg,
                         f_mid=A[1]*T[1]/tau_avg,
                         f_slow=A[2]*T[2]/tau_avg,
                         tau_avg=tau_avg, tau_int=tau_int))
        i += 1

df = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(df.round(3))

# ---- plotting helpers ----
xt = [f"{g}\n{c}" for g, c in zip(df.group, df.cond)]
x = np.arange(len(df))
gcolors = {'3res': 'tab:blue', '9res': 'tab:orange', '9res (spacers)': 'tab:green'}
colors = [gcolors[g] for g in df.group]

# group separators
bounds = np.cumsum([len(l) for _, l in groups])[:-1] - 0.5

# ---- Figure 1: the time constants themselves ----
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, col, name in zip(axes, ['T_fast', 'T_mid', 'T_slow'],
                         ['T fast', 'T mid', 'T slow']):
    ax.bar(x, df[col], color=colors)
    ax.set_ylabel(name)
    for b in bounds:
        ax.axvline(b, color='k', lw=0.8, ls='--')
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(xt, rotation=60, ha='right', fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in gcolors.values()]
axes[0].legend(handles, gcolors.keys(), ncol=3, loc='upper right')
fig.suptitle('Time constants (sorted fast → slow)')
fig.tight_layout()

# ---- Figure 2: share of tau_avg from each component ----
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x, df.f_fast, label='A·T fast')
ax.bar(x, df.f_mid, bottom=df.f_fast, label='A·T mid')
ax.bar(x, df.f_slow, bottom=df.f_fast + df.f_mid, label='A·T slow')
for b in bounds:
    ax.axvline(b, color='k', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels(xt, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('fraction of τ_avg')
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:

# ============================================================
# FIGURE
# ============================================================
fig,axes = plt.subplots(1,1,figsize=(3.3,1.5),sharey=True
                       )

# top row = 3 correlation plots
ax_bar =  axes

# ============================================================
# BOTTOM COMPARISON BAR PLOT
# ============================================================
ax_bar.bar(
    all_labels,
    all_tau,
    color=CLASS_COLOR.values()
)
ax_bar.set_ylabel(r'$\tau_{Protein}$ [ns]')
#ax_bar.set_title('Comparison of Average Correlation Time')


# ============================================================
# GROUP SEPARATOR (3res | 9res | spacer)
# ============================================================

# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

ax_bar.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
ax_bar.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
# ============================================================
# FINAL LAYOUT
# ============================================================
plt.tight_layout()
plt.savefig('Figures/cluster_correlation.png',dpi=500)
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SYSTEMS
# ============================================================

systems = ['3E','3C','3A','3P','3N',
           '9E','9C','9C','9P','9N',
           'X2','X2S','X3','X3S']

# ============================================================
# COLLECT MEAN PERCOLATION VALUES
# ============================================================

perc_values = []

for label in systems:

    df = pd.read_csv(
        f'/Volumes/potoyan-lab/will/gxg-project/new_plots/data/net_metrics_{label}.csv'
    )

    perc_values.append(df['percolation_pc'].mean())

# ============================================================
# COLOR MAP
# ============================================================

# color_map = {
#     '3E': '#D62728',
#     '3A': '#9467BD',
#     '3N': '#FF7F0E',
#     '3P': '#2CA02C',

#     '9E': '#D62728',
#     '9A': '#9467BD',
#     '9N': '#FF7F0E',
#     '9C': '#1F77B4',

#     'X2' : '#8C564B',
#     'X2S': '#E377C2',
#     'X3' : '#7F7F7F',
#     'X3S': '#BCBD22'
# }

#colors = [color_map[s] for s in systems]

# ============================================================
# BAR PLOT
# ============================================================
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"]  = ["Times New Roman"]
fig, ax = plt.subplots(figsize=(3.3,1.5),sharey=True)

ax.bar(systems,
       perc_values,
        color=CLASS_COLOR.values()
      )

# ============================================================
# SEPARATOR LINES
# ============================================================

ax.axvline(4.5, linestyle='--', linewidth=1)
ax.axvline(8.5, linestyle='--', linewidth=1)

# ============================================================
# LABELS
# ============================================================

ax.set_ylabel('Mean Percolation $P_c$')
#ax.set_xlabel('Systems')
plt.ylim(0.60,0.85)
plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.savefig('Figures/mean_percolation.png',dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

y = np.array(all_tau_protein)
x = np.array(perc_values)

labels_x = np.array(all_labels_protein)
labels_y = np.array(systems)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# COLORS
# ============================================================

point_colors = [CLASS_COLOR[s] for s in labels_x]

# ============================================================
# LABEL OFFSETS (in points)
# ============================================================

NUDGE = {
    "X2":  (10, -5),
    "X2S": (10, -5),
    "9E":  ( -10,  8),
    "9C":  ( -8, 5),
    "9A":  (-10,   0),
    "9P":  (10, -5),
    "9N":  (10, -5),
    "X3":  (-10,  -8),
    "X3S": ( 0, -8),
    "3C":  ( 10, 12),
    "3A":  (10, -5),
    "3P":  ( 8, 6),
    "3E":  (-10, 5),
    "3N":  (4,15),
}

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.5, 2.4))

# ------------------------------------------------------------
# Scatter points
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=50,
    edgecolor='black',
    color=point_colors,
    linewidth=0.5,
    zorder=3,
)

# ------------------------------------------------------------
# Labels + connector lines
# ------------------------------------------------------------

for i in range(len(x)):

    label = labels_x[i]

    dx, dy = NUDGE.get(label, (10, 10))

    ax.annotate(
        label,

        xy=(x[i], y[i]),              # point location
        xytext=(dx, dy),              # label offset

        textcoords='offset points',

        fontsize=6,
        fontweight='bold',

        color=CLASS_COLOR[label],

        ha='center',
        va='center',

        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle='-',
            color='gray',
            lw=0.7,
            alpha=0.8,
            shrinkA=0,
            shrinkB=4,
        ),

        zorder=4,
    )

# ------------------------------------------------------------
# Linear fit
# ------------------------------------------------------------

ax.plot(
    x_fit,
    y_fit,
    color='black',
    linewidth=1.5,
    zorder=1,
)

# ============================================================
# LEGEND
# ============================================================

class_handles = [

#     Line2D(
#         [0], [0],
#         marker='o',
#         color='w',
#         markerfacecolor='#1f77b4',
#         markeredgecolor='#1f77b4',
#         markersize=6,
#         label='3-residue',
#     ),

#     Line2D(
#         [0], [0],
#         marker='o',
#         color='w',
#         markerfacecolor='#2ca02c',
#         markeredgecolor='#2ca02c',
#         markersize=6,
#         label='9-residue',
#     ),

#     Line2D(
#         [0], [0],
#         marker='o',
#         color='w',
#         markerfacecolor='#d62728',
#         markeredgecolor='#d62728',
#         markersize=6,
#         label='patterned',
#     ),

    Line2D(
        [0], [0],
        color='black',
        linewidth=1.5,
        label=f'R = {r_value:.2f}',
    ),
]

ax.legend(
    handles=class_handles,
    frameon=False,
    fontsize=5,
    loc='best',
)

# ============================================================
# STYLE
# ============================================================

ax.set_ylabel(
r'$\tau_{Protein}$ [ns]',
    fontsize=7,
)

ax.set_xlabel(
    r'Mean Percolation $P_c$',
    fontsize=7,
)

ax.tick_params(
    axis='both',
    labelsize=7,
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    'Figures/percola_vs_protein.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

In [ ]:

direct_conts = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_9_conts_timeseries.csv', 
                           index_col=0, low_memory=False)

close_spread = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/close_spread_conts_timeseries.csv', 
                           index_col=0, low_memory=False)
direct_conts = pd.concat([direct_conts, close_spread], axis=1)

# direct_conts = direct_conts.rename(columns={'3_E':'3E', '3_C':'3C', '3_A':'3A', '3_P':'3P',
#        '3_N':'3N', '9_E':'9E', '9_C':'9C', '9_A':'9A',
#        '9_P':'9P', '9_N':'9N', '2_close':'GGGXGXGGG','2_spread':'GXGGGGGXG', '3_close':'GGGXXXGGG',
#        '3_spread':'GXGGXGGXG'})

direct_conts = direct_conts.rename(columns={'3_E':'3E', '3_C':'3C', '3_A':'3A', '3_P':'3P',
       '3_N':'3N', '9_E':'9E', '9_C':'9C', '9_A':'9A',
       '9_P':'9P', '9_N':'9N', '2_close':'X2','2_spread':'X2S', '3_close':'X3',
       '3_spread':'X3S'})


direct_conts = direct_conts[['3E','3A','3N','3P','3C',
           '9E','9A','9N','9P','9C',
           'X2','X2S','X3','X3S']]
n_res_dict = {'3E': 900,'3C': 870,'3A': 909,'3P': 870,'3N': 876,'9E': 900,'9C': 855,'9A': 756,'9P': 855,
 '9N': 738,'X2': 720,'X2S': 720,'X3': 720,'X3S': 720}

all_conts = pd.DataFrame()
for i,j in n_res_dict.items():
    for k in direct_conts.columns:
        if i == k:
            all_conts.loc['Direct',f'{k}'] = direct_conts[k].mean() / n_res_dict[k]


In [ ]:
systems = all_conts.columns
values = all_conts.loc['Direct']

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(3.8,1.7))

ax.bar(systems,
       values,
       color=CLASS_COLOR.values())

# ============================================================
# GROUP SEPARATORS
# ============================================================

ax.axvline(4.5, linestyle='--', linewidth=1)
ax.axvline(9.5, linestyle='--', linewidth=1)

# ============================================================
# LABELS
# ============================================================

ax.set_ylabel('Contacts per residue')
ax.set_xlabel('Systems')
plt.ylim(6,8)
plt.tick_params(axis='x', which='minor', bottom=False, top=False)

plt.tight_layout()
plt.show()

In [ ]:
all_tau_contact = np.array(values)
all_labels_contact = systems 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

x = np.array(all_tau_protein)
y = np.array(all_tau_contact)

labels_x = np.array(all_labels_protein)
labels_y = np.array(all_labels_contact)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

y_fit = slope * x + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(3,2))

# scatter
ax.scatter(
    x,
    y,
    s=50,
    edgecolor='black',
    linewidth=0.8,
)

# annotate points
for i in range(len(x)):
    ax.text(
        x[i],
        y[i],
        labels_x[i],   # or labels_y[i]
        fontsize=6,
        ha='left',
        va='bottom'
    )

# linear fit
ax.plot(
    x,
    y_fit,
    linewidth=2,
    #label=f'y = {slope:.2f}x + {intercept:.2f}\nR = {r_value:.2f}',
    label=f'R = {r_value:.2f}'
)

# ============================================================
# STYLE
# ============================================================

ax.set_xlabel(r'Protein $\tau$ (ns)', fontsize=7)
ax.set_ylabel('Water / Protein Mass Ratio', fontsize=7)

ax.tick_params(axis='both', labelsize=7)

ax.legend(frameon=False, fontsize=5)

# cleaner look
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('Figures/water_mass_vs_protein.png',dpi=500)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# -------------------------------------------------------
# Config
# -------------------------------------------------------
ORDER = ["3E", "3C", "3A", "3P", "3N",
         "9E", "9C", "9A", "9P", "9N",
         "X2", "X2S", "X3", "X3S"]
CLASS_COLOR = {
    **{s: "#1f77b4" for s in ["3E", "3C", "3A", "3P", "3N"]},
    **{s: "#2ca02c" for s in ["9E", "9C", "9A", "9P", "9N"]},
    **{s: "#d62728" for s in ["X2", "X2S", "X3", "X3S"]},
}
HL = {}

def col(s):
    return HL.get(s, CLASS_COLOR[s])

NUDGE = {
    "X2":  (-10,   8),
    "X2S": ( 8,   8),
    "9E":  ( 10,  -8),
    "9C":  ( 0,   10),
    "9A":  (-10,   0),
    "9P":  (-10,  8),
    "9N":  (-10, 10),
    "X3":  (10,  -8),
    "X3S": ( -10,   2),
    "3C":  ( 0,   8),
    "3A":  ( 10,  -8),
    "3P":  ( 8, 10),
    "3E":  (0,  8),
    "3N":  (4,15),
}

# -------------------------------------------------------
# Means from DataFrames
# -------------------------------------------------------
size_mean = total_size_df.mean()
sasa_mean = total_sasa_df.mean()

# -------------------------------------------------------
# Align all three quantities by system label
# -------------------------------------------------------
tau_series = pd.Series(dict(zip(all_labels_protein, all_tau_protein)))

common_systems = (
    tau_series.index
    .intersection(size_mean.index)
    .intersection(sasa_mean.index)
)
common_systems = [s for s in ORDER if s in common_systems]

tau_vals  = np.array([tau_series[s] for s in common_systems])
size_vals = np.array([size_mean[s]  for s in common_systems])
sasa_vals = np.array([sasa_mean[s]  for s in common_systems])
labels    = np.array(common_systems)

# -------------------------------------------------------
# Standardize features (so a, b are comparable in magnitude)
# -------------------------------------------------------
scaler = StandardScaler()
X_raw    = np.column_stack([sasa_vals, size_vals])   # shape (n, 2): [SASA, cluster_size]
X_scaled = scaler.fit_transform(X_raw)               # z-score each feature

# -------------------------------------------------------
# Multiple linear regression: tau = a·SASA + b·cluster_size + c
# -------------------------------------------------------
reg = LinearRegression()
reg.fit(X_scaled, tau_vals)

a, b   = reg.coef_                  # weights for [SASA, cluster_size]
c      = reg.intercept_             # intercept
tau_pred = reg.predict(X_scaled)    # fitted tau values

r2     = r2_score(tau_vals, tau_pred)
r, p   = pearsonr(tau_pred, tau_vals)

# Combined score: the single scalar a·SASA_z + b·size_z
# (the regression prediction minus the intercept)
combined = X_scaled[:, 0] * a + X_scaled[:, 1] * b   # == tau_pred - c

# -------------------------------------------------------
# Print fit info
# -------------------------------------------------------
print(f"{'='*50}")
print(f"  Multiple Linear Regression")
print(f"  τ = a·SASA(z) + b·cluster_size(z) + c")
print(f"{'='*50}")
print(f"  a (SASA weight)         : {a:.4f}")
print(f"  b (cluster_size weight) : {b:.4f}")
print(f"  c (intercept)           : {c:.4f}")
print(f"  R²                      : {r2:.4f}")
print(f"  R (Pearson)             : {r:.4f}")
print(f"  p-value                 : {p:.4e}")

# -------------------------------------------------------
# Diagnostic table
# -------------------------------------------------------
diag = pd.DataFrame({
    "tau (ns)"       : tau_vals,
    "SASA (z)"       : X_scaled[:, 0],
    "cluster_size (z)": X_scaled[:, 1],
    "combined"       : combined,
    "tau_predicted"  : tau_pred,
}, index=labels)
print(f"\n{diag.round(3)}\n")

# -------------------------------------------------------
# Plot: combined score  vs  tau
# -------------------------------------------------------
# Re-fit a simple linregress on combined vs tau for the line + R
from scipy.stats import linregress
slope_1d, intercept_1d, r_1d, p_1d, se_1d = linregress(combined, tau_vals)
fit_line = slope_1d * np.sort(combined) + intercept_1d

fig, ax = plt.subplots(figsize=(2.3, 2.4))

ax.plot(
    np.sort(combined), fit_line,
    color="black", linewidth=1.5, zorder=1,
)

for i, s in enumerate(labels):
    ax.scatter(
        combined[i], tau_vals[i],
        s=50,
        color=col(s),
        edgecolor="black",
        linewidth=0.7,
        zorder=3,
    )
    dx, dy = NUDGE.get(s, (0.0, 0.0))
#     ax.text(
#         combined[i] + dx, tau_vals[i] + dy, s,
#         fontsize=6,
#         ha="center", va="center",
#         color=col(s),
#         fontweight="bold",
#         bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=0.4),
#         zorder=4,
#     )
    
    
    # label with connector line
    ax.annotate(
        s,
        xy=(combined[i], tau_vals[i]),                 # point location
        xytext=(dx, dy),           # text offset
        textcoords="offset points",
        fontsize=6,
        fontweight="bold",
        color=col(s),
        ha="center",
        va="center",

        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle="-",        # straight connector line
            color="gray",
            lw=0.7,
            shrinkA=0,
            shrinkB=4,
            alpha=0.8,
        ),

        zorder=4,
    )


# ---- legend ----
class_handles = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor="#1f77b4", markeredgecolor="#1f77b4",
           markersize=6, label="3-residue"),
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor="#2ca02c", markeredgecolor="#2ca02c",
           markersize=6, label="9-residue"),
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor="#d62728", markeredgecolor="#d62728",
           markersize=6, label="patterned"),
#     Line2D([0], [0], color="black", linewidth=1.5,
#            label=f"R = {r_1d:.2f}, p = {p_1d:.2e}"),
    Line2D([0], [0], color="black", linewidth=1.5,
           label=f"R = {r_1d:.2f}"),
    

]
ax.legend(handles=class_handles, fontsize=5, frameon=False, loc="best")

ax.set_xlabel(
    f"$a$·SASA + $b$·cluster size\n"
    f"($a$={a:.2f}, $b$={b:.2f}, standardized)",
    fontsize=7,
)
ax.set_ylabel(r'$\tau_{Protein}$ [ns]', fontsize=7)
ax.tick_params(axis="both", labelsize=7)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figures/mlr_combined_vs_tau_protein.png", dpi=500)
plt.show()

# Figure - 3

In [ ]:

total_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)
        
        
for i in ['9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water_9res.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    num_stickers = 1

    df = pd.read_csv(f'data/{i}_cluster_water_9res.csv')
    
    ratio = df['water_mass'] / df['protein_mass']
    
    temp_df = pd.DataFrame()
    temp_df[name] = ratio.values / num_stickers   # normalize like size
    
    total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# Rename columns (same as size)
# -------------------------
total_ratio_df = total_ratio_df.rename(columns={
    '3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P', '3N':'3N',
    '9E':'9E', '9C':'9C', '9A':'9A', '9P':'9P', '9N':'9N',
    'GGGXGXGGG':'X2', 'GXGGGGGXG':'X2S',
    'GGGXXXGGG':'X3', 'GXGGXGGXG':'X3S'
})


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# BOOTSTRAP MEAN + ERROR ESTIMATION
# ============================================================

n_bootstrap = 500   # number of bootstrap resamples
sample_size = 4      # take 4 samples each time

bootstrap_means = {}

# Loop over each system/column
for col in total_ratio_df.columns:

    data = total_ratio_df[col].dropna().values

    boot_means = []

    for _ in range(n_bootstrap):

        # randomly choose 4 samples WITH replacement
        sample = np.random.choice(data,
                                  size=sample_size,
                                  replace=True)

        boot_means.append(np.mean(sample))

    bootstrap_means[col] = boot_means

# Convert to DataFrame
bootstrap_df = pd.DataFrame(bootstrap_means)

# Mean and bootstrap error
ratio_mean = bootstrap_df.mean()
ratio_std  = bootstrap_df.std()

# ============================================================
# PLOT
# ============================================================

x = np.arange(len(ratio_mean))

plt.figure(figsize=(3.0, 1.6))

plt.bar(
    x,
    ratio_mean,
    yerr=ratio_std,
    width=0.6,
    color=CLASS_COLOR.values(),
    capsize=2
)

# ============================================================
# GROUP SEPARATOR (3res | 9res | spacer)
# ============================================================

# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

plt.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
plt.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)
########

# Labels
plt.xticks(x, ratio_mean.index)
plt.ylabel('Water / Protein Mass Ratio')
#plt.xlabel('System')
plt.grid(axis='y', alpha=0.3)
plt.ylim(0.6, 1.2)

plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.xticks(rotation=45)
plt.savefig('Figures/water_number.png',dpi=400)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# BOOTSTRAP MEAN + ERROR ESTIMATION
# ============================================================

n_bootstrap = 500
sample_size = 4

bootstrap_means = {}

# ------------------------------------------------------------
# Bootstrap distributions
# ------------------------------------------------------------

for col in total_ratio_df.columns:

    data = total_ratio_df[col].dropna().values

    boot_means = []

    for _ in range(n_bootstrap):

        sample = np.random.choice(
            data,
            size=sample_size,
            replace=True
        )

        boot_means.append(np.mean(sample))

    bootstrap_means[col] = boot_means

# Convert to DataFrame
bootstrap_df = pd.DataFrame(bootstrap_means)

# ============================================================
# ORDER + COLORS
# ============================================================

systems = list(bootstrap_df.columns)

x = np.arange(1, len(systems) + 1)

colors = [CLASS_COLOR[s] for s in systems]

# ============================================================
# VIOLIN PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(3.0, 1.6))

# violin data
violin_data = [bootstrap_df[s].values for s in systems]

parts = ax.violinplot(
    violin_data,
    positions=x,
    widths=0.7,
    showmeans=False,
    showmedians=True,
    showextrema=False,
)

# ------------------------------------------------------------
# Color each violin
# ------------------------------------------------------------

for body, c in zip(parts['bodies'], colors):

    body.set_facecolor(c)
    body.set_edgecolor('black')
    body.set_alpha(0.8)
    body.set_linewidth(0.5)

# median line
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(1.0)

# ============================================================
# Overlay mean ± std
# ============================================================

ratio_mean = bootstrap_df.mean()
ratio_std  = bootstrap_df.std()

ax.errorbar(
    x,
    ratio_mean.values,
    yerr=ratio_std.values,
    fmt='o',
    color='black',
    markersize=2.5,
    linewidth=0.8,
    capsize=2,
    zorder=3,
)

# ============================================================
# GROUP SEPARATORS
# ============================================================

group_3_end = 5.5
group_9_end = 10.5

ax.axvline(
    group_3_end,
    color='grey',
    linestyle='--',
    linewidth=0.6,
    alpha=0.7,
)

ax.axvline(
    group_9_end,
    color='grey',
    linestyle='--',
    linewidth=0.6,
    alpha=0.7,
)

# ============================================================
# STYLE
# ============================================================

ax.set_xticks(x)
ax.set_xticklabels(
    systems,
    rotation=45,
    fontsize=7,
)

ax.set_ylabel(
    'Water / Protein Mass Ratio',
    fontsize=7,
)

ax.tick_params(
    axis='y',
    labelsize=7,
)

ax.grid(
    axis='y',
    alpha=0.3,
)

ax.set_ylim(0.6, 1.2)

plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.xticks(rotation=45)

plt.savefig(
    'Figures/water_number_violin.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

y = np.array(all_tau_protein)
x = np.array(ratio_mean)

labels_y = np.array(all_labels_protein)
labels_x = np.array(ratio_mean.index)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# COLORS
# ============================================================

point_colors = [CLASS_COLOR[s] for s in labels_x]

# ============================================================
# LABEL OFFSETS (in points)
# ============================================================

NUDGE = {
    "X2":  (18,  10),
    "X2S": ( 18,  -5),
    "9E":  ( 0, -8),
    "9C":  ( 16,  10),
    "9A":  (0,  8),
    "9P":  (8, 10),
    "9N":  (-18, -12),
    "X3":  (-18, -10),
    "X3S": ( 18,  0),
    "3C":  ( 16,  10),
    "3A":  ( -16, -10),
    "3P":  ( 16, -12),
    "3E":  (16, 10),
    "3N":  (-18, -12),
}

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.6, 2.0))

# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=50,
    color=point_colors,
    edgecolor='black',
    linewidth=0.7,
    zorder=3,
)

# ------------------------------------------------------------
# Labels + connector lines
# ------------------------------------------------------------

for i in range(len(x)):

    label = labels_x[i]

    dx, dy = NUDGE.get(label, (10, 10))

    ax.annotate(
        label,

        xy=(x[i], y[i]),
        xytext=(dx, dy),

        textcoords='offset points',

        fontsize=6,
        fontweight='bold',

        color=CLASS_COLOR[label],

        ha='center',
        va='center',

        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle='-',
            color='gray',
            lw=0.7,
            alpha=0.8,
            shrinkA=0,
            shrinkB=4,
        ),

        zorder=4,
    )

# ------------------------------------------------------------
# Linear fit
# ------------------------------------------------------------

ax.plot(
    x_fit,
    y_fit,
    color='black',
    linewidth=1.5,
    zorder=1,
)

# ============================================================
# LEGEND
# ============================================================

class_handles = [

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#1f77b4',
        markeredgecolor='#1f77b4',
        markersize=6,
        label='3-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#2ca02c',
        markeredgecolor='#2ca02c',
        markersize=6,
        label='9-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#d62728',
        markeredgecolor='#d62728',
        markersize=6,
        label='patterned',
    ),

    Line2D(
        [0], [0],
        color='black',
        linewidth=1.5,
        label=f'R = {r_value:.2f}',
    ),
]

ax.legend(
    handles=class_handles,
    frameon=False,
    fontsize=5,
    loc='best',
)

# ============================================================
# STYLE
# ============================================================

ax.set_ylabel(
    r'$\tau_{protein}$ (ns)',
    fontsize=7,
)

ax.set_xlabel(
    'Water / Protein Mass Ratio',
    fontsize=7,
)

ax.tick_params(
    axis='both',
    labelsize=7,
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    'Figures/water_mass_vs_protein.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

In [ ]:
## Inner -- outer water 

import pandas as pd

total_layer_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)
        
        
for i in ['9res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water_{i}.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'X2'
    elif i == '2_spread':
        name = 'X2S'
    elif i == '3_close':
        name = 'X3'
    else:
        name = 'X3S'
    
    df = pd.read_csv(f'data/{i}_inner_outer_water_9res.csv')
    
    inner_ratio = df['inner_water_unique'] / df['total_water']
    outer_ratio = df['outer_water'] / df['total_water']
    
    temp_df = pd.DataFrame()
    temp_df[f'{name}_in']  = inner_ratio.values
    temp_df[f'{name}_out'] = outer_ratio.values
    
    total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)
    

# ============================================================
# SELECT COLUMNS
# ============================================================

inner_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_in')]
outer_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_out')]

labels = [col.replace('_in', '') for col in inner_cols]

# ============================================================
# BOOTSTRAP SETTINGS
# ============================================================

n_bootstrap = 1000
sample_size = 4

# ============================================================
# BOOTSTRAP FUNCTION
# ============================================================

def bootstrap_mean_std(df, columns, sample_size=4, n_bootstrap=1000):

    bootstrap_results = {}

    for col in columns:

        data = df[col].dropna().values

        boot_means = []

        for _ in range(n_bootstrap):

            # Sample WITH replacement
            sample = np.random.choice(
                data,
                size=sample_size,
                replace=True
            )

            boot_means.append(np.mean(sample))

        bootstrap_results[col] = boot_means

    bootstrap_df = pd.DataFrame(bootstrap_results)

    mean = bootstrap_df.mean()
    std  = bootstrap_df.std()

    return mean, std


In [ ]:
# ============================================================
# COMPUTE BOOTSTRAP MEAN + ERROR
# ============================================================

inner_mean, inner_std = bootstrap_mean_std(
    total_layer_ratio_df,
    inner_cols,
    sample_size=sample_size,
    n_bootstrap=n_bootstrap
)

# ============================================================
# PLOT
# ============================================================

x = np.arange(len(labels))
width = 0.6

fig, axes = plt.subplots(
    1, 1,
    figsize=(3.1, 1.8),
    sharex=True
)

# ----------------------
# Top: Inner water
# ----------------------
axes.bar(
    x,
    inner_mean,
    width,
    yerr=inner_std,
    capsize=2,
    color=CLASS_COLOR.values()
)

# ============================================================
# GROUP SEPARATORS
# ============================================================

group_3_end = 4.5
group_9_end = 9.5

axes.axvline(
    group_3_end,
    color='grey',
    linestyle='--',
    linewidth=0.6,
    alpha=0.7,
)

axes.axvline(
    group_9_end,
    color='grey',
    linestyle='--',
    linewidth=0.6,
    alpha=0.7,
)



axes.set_ylabel('Inner Water Fraction')
#axes.set_title('Inner Water Distribution')

axes.grid(axis='y', alpha=0.3)

axes.set_xticks(x)
axes.set_xticklabels(labels, rotation=45, ha='right')
axes.minorticks_off()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.xticks(rotation=45)
plt.savefig('Figures/inner_water.png',dpi=500)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

y = np.array(all_tau_protein)
x = np.array(inner_mean)

labels = np.array(all_labels_protein)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# COLORS
# ============================================================

point_colors = [CLASS_COLOR[s] for s in labels]

# ============================================================
# LABEL OFFSETS (in points)
# ============================================================

NUDGE = {
    "X2":  (-18,  10),
    "X2S": ( 0,  -10),
    "9E":  ( 0, -10),
    "9C":  ( 0,  10),
    "9A":  (9, 0),
    "9P":  (16, -10),
    "9N":  (15, -10),
    "X3":  (-18, -10),
    "X3S": (14, 0),
    "3C":  ( 16,  -8),
    "3A":  ( 16, -10),
    "3P":  ( 16, -12),
    "3E":  (16, -10),
    "3N":  (18, -5),
}

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.3, 1.9))

# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=50,
    color=point_colors,
    edgecolor='black',
    linewidth=0.7,
    zorder=3,
)

# ------------------------------------------------------------
# Labels + connector lines
# ------------------------------------------------------------

for i in range(len(x)):

    label = labels[i]

    dx, dy = NUDGE.get(label, (10, 10))

    ax.annotate(
        label,

        xy=(x[i], y[i]),
        xytext=(dx, dy),

        textcoords='offset points',

        fontsize=5,
        fontweight='bold',

        color=CLASS_COLOR[label],

        ha='center',
        va='center',

        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle='-',
            color='gray',
            lw=0.7,
            alpha=0.8,
            shrinkA=0,
            shrinkB=4,
        ),

        zorder=4,
    )

# ------------------------------------------------------------
# Linear fit
# ------------------------------------------------------------

ax.plot(
    x_fit,
    y_fit,
    color='black',
    linewidth=1.5,
    zorder=1,
)

# ============================================================
# LEGEND
# ============================================================

class_handles = [

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#1f77b4',
        markeredgecolor='#1f77b4',
        markersize=6,
        label='3-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#2ca02c',
        markeredgecolor='#2ca02c',
        markersize=6,
        label='9-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#d62728',
        markeredgecolor='#d62728',
        markersize=6,
        label='patterned',
    ),

    Line2D(
        [0], [0],
        color='black',
        linewidth=1.5,
        label=f'R = {r_value:.2f}',
    ),
]

ax.legend(
    handles=class_handles,
    frameon=False,
    fontsize=5,
    loc='best',
)

# ============================================================
# STYLE
# ============================================================

ax.set_xlabel(
    'Inner Water Fraction',
    fontsize=7,
)

ax.set_ylabel(
    r' $\tau_{protein}$ (ns)',
    fontsize=7,
)

ax.tick_params(
    axis='both',
    labelsize=7,
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    'Figures/inner_water_vs_protein.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

### water structure 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

systems = [
    'charged', 'equal', 'polar', 'nonpolar', 'aromatic',
#     '2_close', '2_spread', '3_close', '3_spread'
]
labels = [
    'Charged', 'Equal', 'Polar', 'Nonpolar', 'Aromatic',
#     '2-Close', '2-Spread', '3-Close', '3-Spread'
]
labels2 = [
    '9C', '9E', '9P', '9N', '9A',
#     'X2', 'X2S', 'X3', 'X3S'
 ]

# ── Load data ────────────────────────────────────────────────────────────────
q_inner_all, q_outer_all, q_bulk_all = [], [], []
for sys in systems:
    q_inner_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_inner.txt')[:, 1]))
    q_outer_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_outer.txt')[:, 1]))
    q_bulk_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_bulk.txt')[:, 1]))

# ── Plot ─────────────────────────────────────────────────────────────────────
x     = np.arange(len(systems))
width = 0.25

fig, ax = plt.subplots(figsize=(2.8,1.6), dpi=550)

ax.bar(x - width, q_inner_all, width, label='Inner', color='#3A7FC1', edgecolor='white', linewidth=0.5)
ax.bar(x,         q_outer_all, width, label='Outer', color='#45B08C', edgecolor='white', linewidth=0.5)
ax.bar(x + width, q_bulk_all,  width, label='Bulk',  color='#E07B4A', edgecolor='white', linewidth=0.5)

# Light horizontal grid behind bars
ax.yaxis.grid(True, color='#cccccc', linewidth=0.6, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(labels2, rotation=25, fontsize=9)
ax.set_ylabel(' Tetrahedral Order', fontsize=8.5)
ax.set_ylim(0.28, 0.54)
ax.tick_params(axis='both', length=0)

ax.legend(frameon=False, fontsize=8,ncol=3,bbox_to_anchor=(1.0, 1.3))
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.tight_layout()
plt.savefig('Figures/tet_op_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("dielectric_constants.csv")
labels2 = [
    '9C', '9E', '9P', '9N', '9A',
#     'X2', 'X2S', 'X3', 'X3S'
 ]

x = np.arange(len(df))
width = 0.25

fig, ax = plt.subplots(figsize=(2.2,1.8), dpi=550)

ax.bar(x - width, df["inner"], width, label="Inner")
ax.bar(x,         df["outer"], width, label="Outer")
ax.bar(x + width, df["bulk"],  width, label="Bulk")

ax.set_xticks(x)
ax.set_xticklabels(labels2, rotation=25) 
ax.yaxis.grid(True, color='#cccccc', linewidth=0.6, linestyle='--')
              
ax.set_ylabel("Dielectric Constant")
ax.legend(frameon=False, fontsize=5,ncol=3,bbox_to_anchor=(1.0, 1.3))
plt.tick_params(axis='x', which='minor', bottom=False, top=False)


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# DATA
# =====================================================

systems = ['charged', 'equal', 'polar', 'nonpolar', 'aromatic']
labels2 = ['9C', '9E', '9P', '9N', '9A']

# Tetrahedral order
q_inner_all, q_outer_all, q_bulk_all = [], [], []

for sys in systems:
    q_inner_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_inner.txt')[:, 1]))
    q_outer_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_outer.txt')[:, 1]))
    q_bulk_all.append(np.mean(np.loadtxt(f'Tet_OP_{sys}_bulk.txt')[:, 1]))

# Dielectric constant
df = pd.read_csv("dielectric_constants.csv")

# =====================================================
# PLOT
# =====================================================

x = np.arange(len(systems))
width = 0.25

fig, axes = plt.subplots(
    1, 2,
    figsize=(4.2, 1.8),
    dpi=550,
    sharex=False
)

# -----------------------------------------------------
# Left panel: Tetrahedral order
# -----------------------------------------------------
ax = axes[0]

ax.bar(x - width, q_inner_all, width,
       label='Inner',
       color='#3A7FC1',
       edgecolor='white',
       linewidth=0.5)

ax.bar(x, q_outer_all, width,
       label='Outer',
       color='#45B08C',
       edgecolor='white',
       linewidth=0.5)

ax.bar(x + width, q_bulk_all, width,
       label='Bulk',
       color='#E07B4A',
       edgecolor='white',
       linewidth=0.5)

ax.set_ylabel('Tetrahedral Order', fontsize=8)
ax.set_ylim(0.28, 0.54)
ax.set_xticks(x)
ax.set_xticklabels(labels2, rotation=25)
ax.yaxis.grid(True, color='#cccccc', linewidth=0.6, linestyle='--')
ax.tick_params(axis='both', length=0)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)

#ax.set_title('A', loc='left', fontweight='bold')

# -----------------------------------------------------
# Right panel: Dielectric constant
# -----------------------------------------------------
ax = axes[1]

ax.bar(x - width, df["inner"], width,
       label='Inner',
       color='#3A7FC1',
       edgecolor='white',
       linewidth=0.5)

ax.bar(x, df["outer"], width,
       label='Outer',
       color='#45B08C',
       edgecolor='white',
       linewidth=0.5)

ax.bar(x + width, df["bulk"], width,
       label='Bulk',
       color='#E07B4A',
       edgecolor='white',
       linewidth=0.5)

ax.set_ylabel('Dielectric Constant', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels2, rotation=25)
ax.yaxis.grid(True, color='#cccccc', linewidth=0.6, linestyle='--')
ax.tick_params(axis='both', length=0)
#ax.set_title('B', loc='left', fontweight='bold')

# -----------------------------------------------------
# Shared legend
# -----------------------------------------------------
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.94),
    ncol=3,
    frameon=False,
    fontsize=6
)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)

plt.tight_layout()
plt.subplots_adjust(top=0.82, wspace=0.35)

plt.savefig(
    'Figures/tetrahedral_dielectric_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:

total_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        # Load the file
        df = pd.read_csv(f'data/{j}_waterbridges_{i}.csv')
        
        # Create temporary DataFrame with proper column name
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = df['WBC_perresidue'].values
        
        # Append to total DataFrame
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)

# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3

    num_stickers = 1
    # Use the correct filename for each system
    df = pd.read_csv(f'data/{i}_waterbridges_9res.csv')
    
    temp_df = pd.DataFrame()
    # Normalize by number of stickers if needed
    temp_df[name] = df['WBC_perresidue'].values / num_stickers
    
    total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)

# -------------------------
# Rename columns (to match size_df)
# -------------------------
total_ratio_df = total_ratio_df.rename(columns={
    '3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P', '3N':'3N',
    '9E':'9E', '9C':'9C', '9A':'9A', '9P':'9P', '9N':'9N',
    'GGGXGXGGG':'X2', 'GXGGGGGXG':'X2S',
    'GGGXXXGGG':'X3', 'GXGGXGGXG':'X3S'
})

total_ratio_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# BOOTSTRAP SETTINGS
# ============================================================

n_bootstrap = 1000
sample_size = 4

# ============================================================
# BOOTSTRAP MEAN + ERROR
# ============================================================

bootstrap_results = {}

for col in total_ratio_df.columns:

    data = total_ratio_df[col].dropna().values

    boot_means = []

    for _ in range(n_bootstrap):

        # Randomly select 4 samples WITH replacement
        sample = np.random.choice(
            data,
            size=sample_size,
            replace=True
        )

        boot_means.append(np.mean(sample))

    bootstrap_results[col] = boot_means

# Convert to DataFrame
bootstrap_df = pd.DataFrame(bootstrap_results)

# Bootstrap statistics
means = bootstrap_df.mean()
stds  = bootstrap_df.std()

# ============================================================
# PLOT
# ============================================================

x = np.arange(len(means))
width = 0.6

plt.figure(figsize=(3.0, 1.6))

plt.bar(
    x,
    means,
    width,
    yerr=stds,
    capsize=2,
    color=CLASS_COLOR.values(),
    alpha=0.8
)

# Labels
plt.xticks(
    x,
    means.index,
    rotation=45,
    ha='right'
)

# ============================================================
# GROUP SEPARATOR (3res | 9res | spacer)
# ============================================================

# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

plt.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
plt.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)
########

plt.ylabel('WaterBridge per residue')
#plt.xlabel('System')
plt.ylim(0.5,2.4)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.minorticks_off()
plt.savefig('Figures/wb.png',dpi=500)

plt.show()

In [ ]:
# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv("condensate_center_rdf_protein_9_charge.csv")
df_ow = pd.read_csv("condensate_center_rdf_water_9_charge.csv")

# =====================================================
# AVERAGE RDF
# =====================================================

avg_rdf = (
    df.groupby("r", as_index=False)["g_r"]
    .mean()
)

avg_rdf_ow = (
    df_ow.groupby("r", as_index=False)["g_r"]
    .mean()
)

# =====================================================
# STYLE
# =====================================================

# plt.rcParams.update({
#     "font.size": 12,
#     "font.family": "sans-serif",
#     "axes.linewidth": 1.5,
#     "xtick.major.width": 1.4,
#     "ytick.major.width": 1.4,
#     "xtick.direction": "in",
#     "ytick.direction": "in",
# })

# =====================================================
# FIGURE
# =====================================================

fig, ax1 = plt.subplots(figsize=(4.5,2.5), dpi=300)

# -----------------------------------------------------
# COLORS
# -----------------------------------------------------

protein_color = "#C44E52"   # deep red
water_color   = "#4C72B0"   # blue

# =====================================================
# LEFT AXIS : PROTEIN RDF
# =====================================================

line1 = ax1.plot(
    avg_rdf["r"],
    avg_rdf["g_r"],
    lw=1.0,
    color=protein_color,
    label="Heavy atoms",
    zorder=3
)

# subtle fill
ax1.fill_between(
    avg_rdf["r"],
    avg_rdf["g_r"],
    alpha=0.15,
    color=protein_color
)

ax1.set_xlabel(r"Distance from condensate center, $r$ ($\AA$)", fontsize=9)
ax1.set_ylabel(r"Protein $g(r)$", fontsize=9, color=protein_color)

ax1.tick_params(axis='y', colors=protein_color)
ax1.tick_params(axis='both', length=6)

# =====================================================
# RIGHT AXIS : WATER RDF
# =====================================================

ax2 = ax1.twinx()

line2 = ax2.plot(
    avg_rdf_ow["r"],
    avg_rdf_ow["g_r"],
    lw=1.0,
    ls='--',
    color=water_color,
    label="Water",
    zorder=2
)

ax2.fill_between(
    avg_rdf_ow["r"],
    avg_rdf_ow["g_r"],
    alpha=0.12,
    color=water_color
)

ax2.set_ylabel(r"Water $g(r)$", fontsize=9, color=water_color)

ax2.tick_params(axis='y', colors=water_color)
ax2.tick_params(axis='both', length=6)

# =====================================================
# GRID + SPINES
# =====================================================

ax1.grid(
    alpha=0.25,
    linestyle='--',
    linewidth=0.5
)

# # cleaner top spines
# ax1.spines['top'].set_visible(False)
# ax2.spines['top'].set_visible(False)

# =====================================================
# LEGEND
# =====================================================

lines = line1 + line2
labels = [l.get_label() for l in lines]

legend = ax1.legend(
    lines,
    labels,
    loc='lower right',
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize=9
)

# =====================================================
# OPTIONAL REFERENCE LINE
# =====================================================

# ax1.axhline(
#     y=1,
#     color='gray',
#     lw=1.2,
#     ls=':',
#     alpha=0.7
# )

# =====================================================
# TITLE
# =====================================================

# plt.title(
#     "Radial Distribution Around Condensate Center",
#     fontsize=14,
#     pad=12,
#     weight='bold'
# )

# =====================================================
# LAYOUT
# =====================================================

fig.tight_layout()

plt.savefig("Figures/rdf_condensate_center.png",
            dpi=600,
            bbox_inches='tight')

plt.show()

In [ ]:
x= ['equal','aromatic','polar','nonpolar']

for i in x:

    df = pd.read_csv(f"condensate_center_rdf_protein_9_{i}.csv")
    df_ow = pd.read_csv(f"condensate_center_rdf_water_9_{i}.csv")
    




In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# SYSTEMS
# =====================================================

systems = ["equal", "aromatic", "polar", "nonpolar"]
titles  = {"equal": "9E", "aromatic": "9A",
           "polar": "9P", "nonpolar": "9N"}

protein_color = "#C44E52"   # deep red
water_color   = "#4C72B0"   # blue

# =====================================================
# LOAD + AVERAGE ALL DATA FIRST
# =====================================================

data = {}
for s in systems:
    df    = pd.read_csv(f"condensate_center_rdf_protein_9_{s}.csv")
    df_ow = pd.read_csv(f"condensate_center_rdf_water_9_{s}.csv")
    data[s] = (
        df.groupby("r", as_index=False)["g_r"].mean(),
        df_ow.groupby("r", as_index=False)["g_r"].mean(),
    )

# common y-limits so panels are directly comparable
p_max = max(d[0]["g_r"].max() for d in data.values()) * 1.05
w_max = max(d[1]["g_r"].max() for d in data.values()) * 1.05

# =====================================================
# FIGURE
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(7.5, 5), dpi=300,
                         sharex=True, sharey=True)

for idx, (ax1, s) in enumerate(zip(axes.flat, systems)):
    avg_rdf, avg_rdf_ow = data[s]
    row, col = divmod(idx, 2)

    # ---------------- protein (left axis) ----------------
    line1 = ax1.plot(avg_rdf["r"], avg_rdf["g_r"], lw=1.0,
                     color=protein_color, label="Heavy atoms", zorder=3)
    ax1.fill_between(avg_rdf["r"], avg_rdf["g_r"],
                     alpha=0.15, color=protein_color)
    ax1.set_ylim(0, p_max)
    ax1.tick_params(axis="y", colors=protein_color)
    ax1.tick_params(axis="both", length=4, labelsize=8)
    ax1.grid(alpha=0.25, linestyle="--", linewidth=0.5)

    # ---------------- water (right axis) -----------------
    ax2 = ax1.twinx()
    line2 = ax2.plot(avg_rdf_ow["r"], avg_rdf_ow["g_r"], lw=1.0, ls="--",
                     color=water_color, label="Water", zorder=2)
    ax2.fill_between(avg_rdf_ow["r"], avg_rdf_ow["g_r"],
                     alpha=0.12, color=water_color)
    ax2.set_ylim(0, w_max)
    ax2.tick_params(axis="y", colors=water_color)
    ax2.tick_params(axis="both", length=4, labelsize=8)

    # ---------------- labels only on outer panels --------
    if col == 0:
        ax1.set_ylabel(r"Protein $g(r)$", fontsize=9, color=protein_color)
    if col == 1:
        ax2.set_ylabel(r"Water $g(r)$", fontsize=9, color=water_color)
    else:
        ax2.tick_params(axis="y", labelright=False)
    if row == 1:
        ax1.set_xlabel(r"Distance from condensate center, $r$ ($\AA$)",
                       fontsize=9)

    # ---------------- panel tag + title ------------------
    ax1.text(0.03, 0.93, titles[s],
             transform=ax1.transAxes, fontsize=9,
             fontweight="bold", va="top")

    if idx == 0:
        handles = line1 + line2

# =====================================================
# SHARED LEGEND
# =====================================================

fig.legend(handles, [h.get_label() for h in handles],
           loc="upper center", ncol=2, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, 1.03))

fig.tight_layout()

os.makedirs("Figures", exist_ok=True)
plt.savefig("Figures/SI_fig5.png",
            dpi=600, bbox_inches="tight")
plt.show()

## Figure 4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ─────────────────────────────────────────────────────────────
# TRI-EXPONENTIAL MODEL
# ─────────────────────────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))


# ─────────────────────────────────────────────────────────────
# FIT FUNCTION
# ─────────────────────────────────────────────────────────────
def fit_exp3(t, y):

    t = np.array(t)
    y = np.array(y)

    mask = np.isfinite(y)
    t = t[mask]
    y = y[mask]

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y,
                        p0=p0,
                        bounds=bounds,
                        maxfev=20000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    tau_eff = A1*T1 + A2*T2 + A3*T3

    return popt, tau_eff



In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# SYSTEMS
# ─────────────────────────────────────────────────────────────
systems_3res = ['equal','charged' , 'aromatic', 'polar', 'nonpolar']
systems_9res = ['equal','charged' , 'aromatic', 'polar', 'nonpolar']
systems_spacers = ['2_close', '2_spread', '3_close', '3_spread']

# ─────────────────────────────────────────────────────────────
# LABEL MAP
# ─────────────────────────────────────────────────────────────
label_map = {

    # 3-res systems
    ('3-res systems', 'charged')  : '3C',
    ('3-res systems', 'equal')    : '3E',
    ('3-res systems', 'aromatic') : '3A',
    ('3-res systems', 'nonpolar') : '3N',
    ('3-res systems', 'polar')    : '3P',

    # 9-res systems
    ('9-res systems', 'charged')  : '9C',
    ('9-res systems', 'equal')    : '9E',
    ('9-res systems', 'aromatic') : '9A',
    ('9-res systems', 'nonpolar') : '9N',
    ('9-res systems', 'polar')    : '9P',

    # spacer systems
    ('Spacer systems', '2_close')  : 'X2',
    ('Spacer systems', '2_spread') : 'X2S',
    ('Spacer systems', '3_close')  : 'X3',
    ('Spacer systems', '3_spread') : 'X3S',
}

# ─────────────────────────────────────────────────────────────
# FIGURE SETUP
# top row  -> correlation decay plots
# bottom   -> comparison bar plot
# ─────────────────────────────────────────────────────────────
fig,axes = plt.subplots(1, 3, figsize=(8, 2),sharey=True)

ax1 = axes[0]
ax2 = axes[1]
ax3 = axes[2]

# ─────────────────────────────────────────────────────────────
# GLOBAL STORAGE
# ─────────────────────────────────────────────────────────────
all_tau = []
all_labels = []
all_colors = all_colors

# ─────────────────────────────────────────────────────────────
# MAIN FUNCTION
# ─────────────────────────────────────────────────────────────
def process(file_prefix, systems, ax_decay, title):

    # consistent colors
    #colors = plt.cm.tab10(np.linspace(0, 1, len(systems)))

    for i, sys in enumerate(systems):

        #color = colors[i]
        
        color = all_colors[i]

        # load data
        df = pd.read_csv(
            f"data/{file_prefix}_{sys}_new.csv"
        )

        # time + correlation
        t = np.arange(len(df["SP_all"]))*0.04
        y = np.array(df["SP_all"])

        # normalize
        y = y / y[0]

        # fit
        popt, tau_eff = fit_exp3(t, y)
        yfit = exp3(t, *popt)

        # pretty labels
        pretty_label = label_map[(title, sys)]

        # ─────────────────────────────
        # RAW DATA
        # ─────────────────────────────
        ax_decay.plot(
            t[::8],
            y[::8],
            '--',
            color=color,
            alpha=0.35
        )

        # ─────────────────────────────
        # FITTED CURVE
        # ─────────────────────────────
        ax_decay.plot(
            t,
            yfit,
            lw=2.5,
            color=color,
            label=pretty_label
        )

        # store for bar plot
        all_tau.append(tau_eff)
        all_labels.append(pretty_label)
        all_colors.append(color)

    # ─────────────────────────────
    # AXIS FORMATTING
    # ─────────────────────────────
    ax_decay.set_title(title, fontsize=11)

    ax_decay.set_xlabel("Time")
    #ax_decay.set_ylabel("C(t)")

    ax_decay.set_ylim(0,0.1)

    ax_decay.legend(
        frameon=False,
        fontsize=8
    )

# ─────────────────────────────────────────────────────────────
# RUN ALL SYSTEMS
# ─────────────────────────────────────────────────────────────
process(
    'sp_all_3res',
    systems_3res,
    ax1,
    "3-res systems"
)
ax1.set_ylabel("C(t)")
process(
    'sp_all_9res',
    systems_9res,
    ax2,
    "9-res systems"
)

process(
    'sp_all_9res',
    systems_spacers,
    ax3,
    "Spacer systems"
)

plt.savefig('Figures/sp.png',dpi=500)


In [ ]:
fig, ax_bar = plt.subplots(figsize=(3.6,1.5))

# ─────────────────────────────────────────────────────────────
# SINGLE COMPARISON BAR PLOT
# ─────────────────────────────────────────────────────────────
ax_bar.bar(
    all_labels,
    all_tau,
    color=CLASS_COLOR.values(),
)

ax_bar.set_ylabel(r'$\tau_{\mathrm{eff}}$ (ns)')
ax_bar.set_ylabel(r'$\tau_{\mathrm{water}}$ (ns)')


# ax_bar.set_title(
#     "Comparison of Effective Correlation Time",
#     fontsize=10
# )

ax_bar.grid(
    axis='y',
    #colors=CLASS_COLOR.values(),
    alpha=0.3
)

# ============================================================
# GROUP SEPARATOR (3res | 9res | spacer)
# ============================================================

# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

plt.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
plt.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

# ─────────────────────────────────────────────────────────────
# FINALIZE
# ─────────────────────────────────────────────────────────────
plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.xticks(rotation=45)


plt.savefig(
    'Figures/sp_corr_bar.png',
    dpi=500,
    bbox_inches='tight'
)

plt.show()

In [ ]:
all_tau_water_sp = all_tau
all_labels_water_sp= all_labels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

x = np.array(all_tau_protein)
y = np.array(all_tau_water_sp)

labels = np.array(all_labels_protein)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# COLORS
# ============================================================

point_colors = [CLASS_COLOR[s] for s in labels]

# ============================================================
# LABEL OFFSETS (in points)
# ============================================================

NUDGE = {
    "X2":  (-18,  10),
    "X2S": ( 18,  10),
    "9E":  ( 16, -10),
    "9C":  ( 16,  10),
    "9A":  (0,  14),
    "9P":  (0, 10),
    "9N":  (-18, -12),
    "X3":  (-18, -10),
    "X3S": ( 0,  -15),
    "3C":  ( 0,  10),
    "3A":  ( 16, -10),
    "3P":  ( -6, 7),
    "3E":  (-5, 10),
    "3N":  (8, 5),
}

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.5, 2.3))

# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=50,
    color=point_colors,
    edgecolor='black',
    linewidth=0.7,
    zorder=3,
)

# ------------------------------------------------------------
# Labels + connector lines
# ------------------------------------------------------------

for i in range(len(x)):

    label = labels[i]

    dx, dy = NUDGE.get(label, (10, 10))

    ax.annotate(
        label,

        xy=(x[i], y[i]),
        xytext=(dx, dy),

        textcoords='offset points',

        fontsize=6,
        fontweight='bold',

        color=CLASS_COLOR[label],

        ha='center',
        va='center',

        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle='-',
            color='gray',
            lw=0.7,
            alpha=0.8,
            shrinkA=0,
            shrinkB=4,
        ),

        zorder=4,
    )

# ------------------------------------------------------------
# Linear fit
# ------------------------------------------------------------

ax.plot(
    x_fit,
    y_fit,
    color='black',
    linewidth=1.5,
    zorder=1,
)

# ============================================================
# LEGEND
# ============================================================

class_handles = [

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#1f77b4',
        markeredgecolor='#1f77b4',
        markersize=6,
        label='3-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#2ca02c',
        markeredgecolor='#2ca02c',
        markersize=6,
        label='9-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#d62728',
        markeredgecolor='#d62728',
        markersize=6,
        label='patterned',
    ),

    Line2D(
        [0], [0],
        color='black',
        linewidth=1.5,
        label=f'R = {r_value:.2f}',
    ),
]

ax.legend(
    handles=class_handles,
    frameon=False,
    fontsize=5,
    loc='best',
)

# ============================================================
# STYLE
# ============================================================

ax.set_xlabel(
    r' $\tau_{Protein}$ (ns)',
    fontsize=7,
)

ax.set_ylabel(
    r'$\tau_{water}$ (ns)',
    fontsize=7,
)

ax.tick_params(
    axis='both',
    labelsize=7,
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    'Figures/sp_vs_protein.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────
import numpy as np
import pickle
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# DATASETS
# ─────────────────────────────────────────────────────────────
datasets = [
    ('corr_results/all_water_corr_result_3res.pkl',
     ['equal', 'charged', 'aromatic', 'polar', 'nonpolar'],
     '3-res systems'),

    ('corr_results/all_water_corr_result_9res.pkl',
     ['equal', 'charged', 'aromatic', 'polar', 'nonpolar'],
     '9-res systems'),

    ('corr_results/all_water_corr_result_9res_2.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     'Spacer systems'),
]

# ─────────────────────────────────────────────────────────────
# LABEL MAP
# ─────────────────────────────────────────────────────────────
label_map = {

    ('3-res systems', 'charged')  : '3C',
    ('3-res systems', 'equal')    : '3E',
    ('3-res systems', 'aromatic') : '3A',
    ('3-res systems', 'nonpolar') : '3N',
    ('3-res systems', 'polar')    : '3P',

    ('9-res systems', 'charged')  : '9C',
    ('9-res systems', 'equal')    : '9E',
    ('9-res systems', 'aromatic') : '9A',
    ('9-res systems', 'nonpolar') : '9N',
    ('9-res systems', 'polar')    : '9P',

    ('Spacer systems', '2_close')  : 'X2',
    ('Spacer systems', '2_spread') : 'X2S',
    ('Spacer systems', '3_close')  : 'X3',
    ('Spacer systems', '3_spread') : 'X3S',
}

# ─────────────────────────────────────────────────────────────
# COLORS
# ─────────────────────────────────────────────────────────────
clrs = ['black','red','blue','green','purple','orange']

plt.rcParams.update({"font.size": 13})

# ─────────────────────────────────────────────────────────────
# FIGURE SETUP
# ─────────────────────────────────────────────────────────────
fig,axes = plt.subplots(1, 3, figsize=(9, 1.8),sharey=True)

ax_corr_0 = axes[0]
ax_corr_1 = axes[1]
ax_corr_2 = axes[2]


# # top row: 3 correlation plots
# ax_corr_0 = plt.subplot2grid((2, 3), (0, 0))
# ax_corr_1 = plt.subplot2grid((2, 3), (0, 1), sharey=ax_corr_0)
# ax_corr_2 = plt.subplot2grid((2, 3), (0, 2), sharey=ax_corr_0)

corr_axes = [ax_corr_0, ax_corr_1, ax_corr_2]

# ─────────────────────────────────────────────────────────────
# GLOBAL STORAGE (for bar plot)
# ─────────────────────────────────────────────────────────────
all_labels = []
all_tau = []
all_colors = []

# ─────────────────────────────────────────────────────────────
# MAIN LOOP (each dataset -> one column)
# ─────────────────────────────────────────────────────────────
for col, (file_path, systems, title) in enumerate(datasets):

    with open(file_path, 'rb') as f:
        results = pickle.load(f)

    ax = corr_axes[col]

    for i, sys in enumerate(systems):

        r = results[sys]

        t = np.array(r['tM'])*2
        c = np.array(r['corrN'])

        label = label_map[(title, sys)]
        color = clrs[i % len(clrs)]

        # raw correlation
        ax.plot(t[::5], c[::5], '--', color=color, alpha=0.25)

        # fit
        try:
            popt, tau = fit_exp3(t, c)
            cfit = exp3(t, *popt)

            ax.plot(t, cfit, '-', color=color, lw=2, label=label)

            all_tau.append(tau)
            all_labels.append(label)
            all_colors.append(color)

        except RuntimeError:
            print(f"Fit failed: {sys}")
            all_tau.append(np.nan)
            all_labels.append(label)
            all_colors.append(color)

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("t [ns]")
    ax.set_ylabel("C(t)")
    ax.legend(frameon=False, fontsize=8)
    ax.set_ylim(0,0.4)

# ─────────────────────────────────────────────────────────────
# BAR PLOT (ALL SYSTEMS COMBINED)
# ─────────────────────────────────────────────────────────────
plt.savefig('Figures/corrla_all.png',dpi=500)


In [ ]:
fig, ax_bar = plt.subplots(figsize=(3.6,1.7))


x = np.arange(len(all_tau))

ax_bar.bar(
    x,
    all_tau,
    color=CLASS_COLOR.values(),
    #edgecolor='black',
    linewidth=0.8
)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(all_labels, rotation=45, ha='right')
ax_bar.set_ylabel(r'$\tau_{\mathrm{Protein-water}}$ [ns]')
#ax_bar.set_title("Relaxation time comparison (all systems)")

# ============================================================
# GROUP SEPARATOR (3res | 9res | spacer)
# ============================================================

# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

plt.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
plt.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

plt.grid(
    axis='y',
    #colors=CLASS_COLOR.values(),
    alpha=0.3
)


# ─────────────────────────────────────────────────────────────
# FINAL LAYOUT
# ─────────────────────────────────────────────────────────────
plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.xticks(rotation=45)
plt.savefig('Figures/corrla_all_bar.png',dpi=500)

plt.show()

In [ ]:
all_tau_water_corr = all_tau
all_labels_water_label = all_labels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import linregress

# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

x = np.array(all_tau_water_sp)
y = np.array(all_tau_water_corr)

labels = np.array(all_labels_water_sp)

# Optional check
assert len(x) == len(y), "Protein and water arrays must have same length"

# ============================================================
# COLORS
# ============================================================

point_colors = [CLASS_COLOR[s] for s in labels]

# ============================================================
# LABEL OFFSETS (in points)
# ============================================================

NUDGE = {
    "X2":  (-18,  10),
    "X2S": ( 18,  10),
    "9E":  ( 0, -7),
    "9C":  ( 8,  0),
    "9A":  (0,  10),
    "9P":  (0, 10),
    "9N":  (-18, 12),
    "X3":  (0, 10),
    "X3S": ( 0,  -10),
    "3C":  ( 0,  10),
    "3A":  ( 16, -10),
    "3P":  ( -6, 7),
    "3E":  (-5, 10),
    "3N":  (0, 15),
}

# ============================================================
# LINEAR FIT
# ============================================================

slope, intercept, r_value, p_value, std_err = linregress(x, y)

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.5, 2.3))

# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=50,
    color=point_colors,
    edgecolor='black',
    linewidth=0.7,
    zorder=3,
)

# ------------------------------------------------------------
# Labels + connector lines
# ------------------------------------------------------------

for i in range(len(x)):

    label = labels[i]

    dx, dy = NUDGE.get(label, (10, 10))

    ax.annotate(
        label,

        xy=(x[i], y[i]),
        xytext=(dx, dy),

        textcoords='offset points',

        fontsize=6,
        fontweight='bold',

        color=CLASS_COLOR[label],

        ha='center',
        va='center',

        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.3,
        ),

        arrowprops=dict(
            arrowstyle='-',
            color='gray',
            lw=0.7,
            alpha=0.8,
            shrinkA=0,
            shrinkB=4,
        ),

        zorder=4,
    )

# ------------------------------------------------------------
# Linear fit
# ------------------------------------------------------------

ax.plot(
    x_fit,
    y_fit,
    color='black',
    linewidth=1.5,
    zorder=1,
)

# ============================================================
# LEGEND
# ============================================================

class_handles = [

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#1f77b4',
        markeredgecolor='#1f77b4',
        markersize=6,
        label='3-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#2ca02c',
        markeredgecolor='#2ca02c',
        markersize=6,
        label='9-residue',
    ),

    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#d62728',
        markeredgecolor='#d62728',
        markersize=6,
        label='patterned',
    ),

    Line2D(
        [0], [0],
        color='black',
        linewidth=1.5,
        label=f'R = {r_value:.2f}',
    ),
]

ax.legend(
    handles=class_handles,
    frameon=False,
    fontsize=5,
    loc='lower right',
)

# ============================================================
# STYLE
# ============================================================

ax.set_xlabel(
    r'$\tau_{water}$ (ns)',
    fontsize=7,
)

ax.set_ylabel(
    r'$\tau_{\mathrm{Protein-water}}$ [ns]',
    fontsize=7,
)

ax.tick_params(
    axis='both',
    labelsize=7,
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    'Figures/corr_vs_sp.png',
    dpi=500,
    bbox_inches='tight',
)

plt.show()

# ============================================================
# PRINT FIT INFO
# ============================================================

print(f"Slope      : {slope:.4f}")
print(f"Intercept  : {intercept:.4f}")
print(f"R-value    : {r_value:.4f}")
print(f"P-value    : {p_value:.4e}")
print(f"Std Error  : {std_err:.4f}")

### Inner vs outer 

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMPUTE + STORE FITS
# ─────────────────────────────────────────────────────────────
def process_systems(file_prefix, systems):

    results = {}

    for sys in systems:

        df = pd.read_csv(f'data/{file_prefix}_{sys}_new.csv')

        t = np.arange(len(df['Inner']))*0.04  # same time for both layers

        sys_out = {}

        for lay in layers:

            y = np.array(df[lay])
            y = y / y[0]   # normalize

            popt, tau_eff, yfit = fit_exp3terms(t, y)

            sys_out[lay] = {
                "tau": tau_eff,
                "y": y,
                "fit": yfit
            }

        results[sys] = sys_out

    return results

# ─────────────────────────────────────────────────────────────
# TRI-EXPONENTIAL MODEL
# ─────────────────────────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))


# ─────────────────────────────────────────────────────────────
# FIT FUNCTION
# ─────────────────────────────────────────────────────────────
def fit_exp3terms(t, y):

    t = np.array(t)
    y = np.array(y)

    mask = np.isfinite(y)
    t = t[mask]
    y = y[mask]

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y,
                        p0=p0,
                        bounds=bounds,
                        maxfev=20000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    tau_eff = A1*T1 + A2*T2 + A3*T3
    
    yfit = exp3(t, *popt)

    return popt, tau_eff,yfit


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────
# SYSTEMS
# ─────────────────────────────────────────────────────────────
systems_3res = ['equal','charged', 'aromatic', 'polar', 'nonpolar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']
layers = ['Inner', 'Outer']

# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
data_3res = process_systems('sp_3res', systems_3res)
data_9res = process_systems('sp_9res', systems_9res)
data_spacers = process_systems('sp_9res', systems_spacers)

datasets = [
    (data_3res, "3res"),
    (data_9res, "9res"),
    (data_spacers, "spacers"),
]
color_map = {
    'charged':   '#d62728',  # red
    'equal':     '#7f7f7f',  # grey
    'aromatic':  '#ff7f0e',  # orange
    'nonpolar':  '#1f77b4',  # blue
    'polar':     '#2ca02c',  # green

    '2_close':   '#9467bd',  # purple
    '2_spread':  '#8c564b',  # brown
    '3_close':   '#e377c2',  # pink
    '3_spread':  '#17becf',  # cyan
}

# ─────────────────────────────────────────────────────────────
# LABEL MAP
# ─────────────────────────────────────────────────────────────
label_map = {

    ('3res', 'charged')  : '3C',
    ('3res', 'equal')    : '3E',
    ('3res', 'aromatic') : '3A',
    ('3res', 'nonpolar') : '3N',
    ('3res', 'polar')    : '3P',

    ('9res', 'charged')  : '9C',
    ('9res', 'equal')    : '9E',
    ('9res', 'aromatic') : '9A',
    ('9res', 'nonpolar') : '9N',
    ('9res', 'polar')    : '9P',

    ('spacers', '2_close')  : 'X2',
    ('spacers', '2_spread') : 'X2S',
    ('spacers', '3_close')  : 'X3',
    ('spacers', '3_spread') : 'X3S',
}

# ─────────────────────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(7.3,3.5))

legend_handles = {}
legend_labels  = {}

# ─────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────
for col, (data, title) in enumerate(datasets):

    ax_in   = axes[0, col]
    ax_out  = axes[1, col]

    systems = list(data.keys())

    for sys in systems:

        color = color_map[sys]

        y_in  = data[sys]['Inner']['y']
        y_out = data[sys]['Outer']['y']

        t = np.arange(len(y_in))*0.04

        # =========================
        # INNER
        # =========================
        ax_in.plot(t[::5], y_in[::5], '--', color=color, alpha=0.25)

        h, = ax_in.plot(
            t,
            data[sys]['Inner']['fit'],
            color=color,
            lw=2
        )

        # store handle + custom label
        legend_handles[(title, sys)] = h
        legend_labels[(title, sys)]  = label_map[(title, sys)]

        # =========================
        # OUTER
        # =========================
        ax_out.plot(t[::5], y_out[::5], '--', color=color, alpha=0.25)

        ax_out.plot(
            t,
            data[sys]['Outer']['fit'],
            color=color,
            lw=2
        )

    # ─────────────────────────────
    # FORMATTING
    # ─────────────────────────────
    ax_in.set_title(f"Inner — {title}")
    ax_out.set_title(f"Outer — {title}")

    ax_in.set_ylabel("C(t)")
    ax_out.set_ylabel("C(t)")

    ax_out.set_xlabel("t")

    ax_in.grid(alpha=0.3)
    ax_out.grid(alpha=0.3)

    ax_in.set_ylim(-0.05, 0.5)
    ax_out.set_ylim(-0.05, 0.5)

# ─────────────────────────────────────────────────────────────
# GLOBAL LEGEND
# ─────────────────────────────────────────────────────────────
fig.legend(
    legend_handles.values(),
    legend_labels.values(),
    loc="upper center",
    ncol=7,
    frameon=False
)
plt.tight_layout(rect=[0, 0, 1, 0.92])

plt.savefig('Figures/sp_in_outer.png',dpi=500)

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# TWO BAR PLOTS : INNER + OUTER
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(7,1.8))

ax_in, ax_out = axes

# ─────────────────────────────────────────────────────────────
# STORAGE
# ─────────────────────────────────────────────────────────────
all_labels = []
tau_inner_all = []
tau_outer_all = []
bar_colors = []

# ─────────────────────────────────────────────────────────────
# COLLECT DATA
# ─────────────────────────────────────────────────────────────
for data, title in datasets:

    for sys in data.keys():

        lbl = label_map[(title, sys)]

        all_labels.append(lbl)

        tau_inner_all.append(data[sys]['Inner']['tau'])
        tau_outer_all.append(data[sys]['Outer']['tau'])

        bar_colors.append(color_map[sys])

# x positions
x = np.arange(len(all_labels))

# ─────────────────────────────────────────────────────────────
# INNER PLOT
# ─────────────────────────────────────────────────────────────
ax_in.bar(
    x,
    tau_inner_all,
    color=CLASS_COLOR.values(),
    #alpha=0.85
)

ax_in.set_title("Inner",fontsize=9)
ax_in.set_ylabel(r'$\tau_{\mathrm{water}}$')

ax_in.set_xticks(x)
ax_in.set_xticklabels(all_labels, rotation=45, ha='right')
#ax_in.minorticks_off()
ax_in.grid(alpha=0.3)
ax_in.tick_params(axis='x', which='minor', bottom=False, top=False)
# ─────────────────────────────────────────────────────────────
# OUTER PLOT
# ─────────────────────────────────────────────────────────────
ax_out.bar(
    x,
    tau_outer_all,
    color=CLASS_COLOR.values(),
    #alpha=0.60
)

ax_out.set_title("Outer",fontsize=9)

ax_out.set_xticks(x)
ax_out.set_xticklabels(all_labels, rotation=45, ha='right')

ax_out.grid(alpha=0.3)


# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

ax_in.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
ax_in.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

ax_out.axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
ax_out.axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

# ─────────────────────────────────────────────────────────────
# FINAL LAYOUT
# ─────────────────────────────────────────────────────────────

plt.tight_layout()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)

plt.savefig('Figures/sp_inner_outer_bar.png',dpi=500)
plt.show()


In [ ]:
# ── Config ─────────────────────────────────────────────────────────
pkl_files = [
    ('./corr_results/water_corr_resultsi_3res.pkl',   '3res'),
    ('./corr_results/water_corr_resultsi_9res.pkl',   '9res'),
    ('./corr_results/water_corr_resultsi_9res_2.pkl', '9res_2'),
]

# ─────────────────────────────────────────────────────────────
# LABEL MAP (use SAME keys as your datasets)
# ─────────────────────────────────────────────────────────────
label_map = {

    ('3res', 'charged')  : '3C',
    ('3res', 'equal')    : '3E',
    ('3res', 'aromatic') : '3A',
    ('3res', 'nonpolar') : '3N',
    ('3res', 'polar')    : '3P',

    ('9res', 'charged')  : '9C',
    ('9res', 'equal')    : '9E',
    ('9res', 'aromatic') : '9A',
    ('9res', 'nonpolar') : '9N',
    ('9res', 'polar')    : '9P',

    ('9res_2', '2_close')  : 'X2',
    ('9res_2', '2_spread') : 'X2S',
    ('9res_2', '3_close')  : 'X3',
    ('9res_2', '3_spread') : 'X3S',
}


fig, axes = plt.subplots(2, 3, figsize=(7, 3.2))

# store legend items only ONCE
legend_handles = {}
legend_labels = {}

for col, (pkl_path, title) in enumerate(pkl_files):

    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)

    ax_in  = axes[0, col]
    ax_out = axes[1, col]

    for label in results.keys():

        res = results[label]
        tM = np.array(res['tM'])
        color = color_map[label]

        short_label = label_map.get((title, label), label)

        # =========================
        # INNER
        # =========================
        y_in = np.array(res['corrN'])
        _, tau_in, yfit_in = fit_exp3terms(tM, y_in)

        # raw (NO label → avoids legend clutter)
        ax_in.plot(tM, y_in, color=color, lw=1.2, alpha=0.25)

        # fit (ONLY this goes to legend)
        h_in, = ax_in.plot(
            tM,
            yfit_in,
            color=color,
            lw=2,
            label=short_label
        )

        # store once
        if short_label not in legend_handles:
            legend_handles[short_label] = h_in

        # =========================
        # OUTER
        # =========================
        y_out = np.array(res['corrN_2'])
        ystd  = np.array(res['corrN_std_2'])

        _, tau_out, yfit_out = fit_exp3terms(tM, y_out)

        ax_out.fill_between(
            tM,
            y_out - ystd,
            y_out + ystd,
            color=color,
            alpha=0.15
        )

        ax_out.plot(tM, y_out, color=color, lw=1.2, alpha=0.25)
        ax_out.plot(tM, yfit_out, color=color, lw=2)

    # formatting
    ax_in.set_title(f"{title} : Inner")
    ax_out.set_title(f"{title} : Outer")

    ax_in.set_ylabel("C(t)")
    ax_out.set_ylabel("C(t)")
    ax_out.set_xlabel("Time")

    ax_in.grid(alpha=0.3)
    ax_out.grid(alpha=0.3)

    # IMPORTANT: remove per-axis legends
    # ax_in.legend()
    # ax_out.legend()

# # =========================
# # GLOBAL LEGEND
# # =========================
fig.legend(
    legend_handles.values(),
    legend_handles.keys(),
    loc="upper center",
    ncol=8,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.savefig('Figures/corr_in_outer.png',dpi=500)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

pkl_files = [
    ('./corr_results/water_corr_resultsi_3res.pkl',   '3res'),
    ('./corr_results/water_corr_resultsi_9res.pkl',   '9res'),
    ('./corr_results/water_corr_resultsi_9res_2.pkl', '9res_2'),
]

label_map = {
    ('3res', 'charged'): '3C',
    ('3res', 'equal'): '3E',
    ('3res', 'aromatic'): '3A',
    ('3res', 'nonpolar'): '3N',
    ('3res', 'polar'): '3P',

    ('9res', 'charged'): '9C',
    ('9res', 'equal'): '9E',
    ('9res', 'aromatic'): '9A',
    ('9res', 'nonpolar'): '9N',
    ('9res', 'polar'): '9P',

    ('9res_2', '2_close'): 'X2',
    ('9res_2', '2_spread'): 'X2S',
    ('9res_2', '3_close'): 'X3',
    ('9res_2', '3_spread'): 'X3S',
}

labels_all = []
tau_inner_all = []
tau_outer_all = []

# ── collect data ─────────────────────────────────────────────
for pkl_path, title in pkl_files:

    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)

    for label, res in results.items():

        short_label = label_map.get((title, label), f"{title}-{label}")

        tM = np.array(res['tM'])*2

        # INNER
        y_in = np.array(res['corrN'])
        _, tau_in, _ = fit_exp3terms(tM, y_in)

        # OUTER
        y_out = np.array(res['corrN_2'])
        _, tau_out, _ = fit_exp3terms(tM, y_out)

        labels_all.append(short_label)
        tau_inner_all.append(tau_in)
        tau_outer_all.append(tau_out)

        
###
ORDER = ["3E", "3C", "3A", "3P", "3N",
         "9E", "9C", "9A", "9P", "9N",
         "X2", "X2S", "X3", "X3S"]

# Get indices corresponding to desired order
idx = [labels_all.index(label) for label in ORDER]

labels_all = [labels_all[i] for i in idx]
tau_inner_all = [tau_inner_all[i] for i in idx]
tau_outer_all = [tau_outer_all[i] for i in idx]
        

# ── PLOT ─────────────────────────────────────────────
x = np.arange(len(labels_all))

fig, ax = plt.subplots(1, 2,  figsize=(7,1.8))

# =========================
# INNER
# =========================
ax[0].bar(x, tau_inner_all, color=CLASS_COLOR.values())
ax[0].set_xticks(x)
ax[0].set_xticklabels(labels_all, rotation=45, ha='right')
ax[0].set_ylabel(r'$\tau_{\mathrm{Protein-water}}$ [ns]',fontsize=9)
ax[0].set_title("Inner Time Constants",fontsize=9)
ax[0].grid(alpha=0.3)
ax[0].tick_params(axis='x', which='minor', bottom=False, top=False)

# =========================
# OUTER
# =========================
ax[1].bar(x, tau_outer_all, color=CLASS_COLOR.values())
ax[1].set_xticks(x)
ax[1].set_xticklabels(labels_all, rotation=45, ha='right')
#ax[1].set_ylabel(r"Outer $\tau$",fontsize=9)
ax[1].set_title("Outer Time Constants",fontsize=9)
ax[1].grid(alpha=0.3)
ax[1].tick_params(axis='x', which='minor', bottom=False, top=False)





# 5 bars for 3_res, 5 bars for 9_res, then 4 spacer bars
group_3_end = 4.5   # between 5th and 6th bar (index 4 and 5)
group_9_end = 9.5   # between 10th and spacer (index 9 and 10)

ax[0].axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
ax[0].axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

ax[1].axvline(group_3_end, color='grey', linestyle='--', linewidth=0.5)
ax[1].axvline(group_9_end, color='grey', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('Figures/corr_inner_outer_bar.png',dpi=500)

plt.show()

In [ ]:
tau_inner_all

In [ ]:

with open('./corr_results/water_corr_resultsi_3res.pkl', 'rb') as f:
    results = pickle.load(f)

In [ ]:
 for label, res in results.items():
        print(res)

### Peptide feature making 

In [ ]:
""""Peptide sequences and stoichiometries for the 14 GXG systems.

Stoichiometries extracted from the packmol mixture.inp files in
gxg_comps/{3_res,9_res}/{equal,charged,aromatic,polar,nonpolar}/ and
gxg_comps/9_res/equal/{2_close,2_spread,3_close,3_spread}/.

Single-letter codes are used internally. Each system is a dict
sequence -> count """

from collections import OrderedDict

# canonical amino acids
AA3to1 = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
          "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
          "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
          "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"}

AA_ALL = list("AGRNDCEQHILKMFPSTWYV")

# --- 3-residue systems ------------------------------------------------------
def _gxg(x): return f"G{x}G"

SYSTEMS = OrderedDict()

# 3E: 20 GXG variants equimolar (5 each used in equal/3res mixture)
SYSTEMS["3E"] = {_gxg(x): 8 for x in AA_ALL}

# 3C: enriched charged (R, K, D, E, H) at count 28; others at 10
charged_set = {"R", "K", "D", "E", "H"}
SYSTEMS["3C"] = {_gxg(x): (28 if x in charged_set else 10) for x in AA_ALL}

# 3A: enriched aromatic (F, Y, W) at 33; others at 12
aromatic_set = {"F", "Y", "W"}
SYSTEMS["3A"] = {_gxg(x): (33 if x in aromatic_set else 12) for x in AA_ALL}

# 3P: enriched polar (C, Q, N, S, T) at 28; others at 10
polar_set = {"C", "Q", "N", "S", "T"}
SYSTEMS["3P"] = {_gxg(x): (28 if x in polar_set else 10) for x in AA_ALL}

# 3N: enriched non-polar (G, P, A, V, L, I, M) at 25; others at 9
nonpolar_set = {"G", "P", "A", "V", "L", "I", "M"}
SYSTEMS["3N"] = {_gxg(x): (25 if x in nonpolar_set else 9) for x in AA_ALL}


# --- 9-residue single-sticker systems --------------------------------------
def _g4xg4(x): return f"GGGG{x}GGGG"

# 9E: equimolar over 20 G4XG4 variants
SYSTEMS["9E"] = {_g4xg4(x): 5 for x in AA_ALL}

# 9C: R, D, E, H, K at 10; others at 3
SYSTEMS["9C"] = {_g4xg4(x): (10 if x in charged_set else 3) for x in AA_ALL}

# 9A: F, W, Y at 11; others at 3
SYSTEMS["9A"] = {_g4xg4(x): (11 if x in aromatic_set else 3) for x in AA_ALL}

# 9P: N, C, Q, S, T at 10; others at 3
SYSTEMS["9P"] = {_g4xg4(x): (10 if x in polar_set else 3) for x in AA_ALL}

# 9N: G, A, I, L, M, P, V at 8; others at 2
SYSTEMS["9N"] = {_g4xg4(x): (8 if x in nonpolar_set else 2) for x in AA_ALL}


# --- patterned multi-sticker (9-residue) -----------------------------------
# Each system: equimolar over 20 X identities at count 4
# X2  (close, 2 stickers at positions 4,6)   = GGGXGXGGG
# X2S (spread, 2 stickers at positions 2,8)  = GXGGGGGXG
# X3  (close, 3 stickers at positions 4,5,6) = GGGXXXGGG
# X3S (spread, 3 stickers at 2,5,8)          = GXGGXGGXG
def _pattern(xpositions, x):
    """positions are 1-indexed in a 9-residue chain."""
    seq = list("GGGGGGGGG")
    for p in xpositions:
        seq[p - 1] = x
    return "".join(seq)

PATTERN_POSITIONS = {
    "X2":  (4, 6),
    "X2S": (2, 8),
    "X3":  (4, 5, 6),
    "X3S": (2, 5, 8),
}
for label, positions in PATTERN_POSITIONS.items():
    SYSTEMS[label] = {_pattern(positions, x): 4 for x in AA_ALL}


# --- helpers ---------------------------------------------------------------
def chain_length(sysname):
    """Length of peptide chains in a system (all chains have the same length)."""
    return len(next(iter(SYSTEMS[sysname])))


def x_positions(sysname):
    """1-indexed positions of non-G residues (sticker positions)."""
    if sysname.startswith("3"):
        return (2,)
    if sysname in PATTERN_POSITIONS:
        return PATTERN_POSITIONS[sysname]
    # other 9-residue single-sticker
    return (5,)


def n_stickers(sysname):
    return len(x_positions(sysname))


def total_peptides(sysname):
    return sum(SYSTEMS[sysname].values())


def total_residues(sysname):
    return sum(len(seq) * n for seq, n in SYSTEMS[sysname].items())


if __name__ == "__main__":
    for s in SYSTEMS:
        print(f"{s:>4}  N={total_peptides(s):>4}  L={chain_length(s)}"
              f"  stickers={n_stickers(s)} at {x_positions(s)}"
              f"  total_res={total_residues(s)}")


# ===========================================================================
# Sequence descriptors for peptide mixtures.
#
# For each system (a dict of sequence -> count) we compute:
#     FCR        — fraction of charged residues (R,K,D,E,H)
#     NCPR_abs   — |net charge per residue|
#     f_aromatic — fraction of aromatic residues (F,Y,W)
#     f_sticker  — fraction of non-G residues (Pappu-style sticker fraction)
#     hydropathy — mean Kyte-Doolittle hydropathy
#     sc_volume  — mean side-chain volume (Å^3, Zamyatnin 1972)
#     kappa_xg   — Das-Pappu κ patterning, treating X (non-G) vs G as the two classes
#     sticker_spread — std-dev of non-G positions / chain length (positional dispersion)
#     n_stickers — mean number of stickers (non-G) per chain
#
# All system-level descriptors are mole-fraction-weighted averages over
# the per-chain values.
# ===========================================================================

import numpy as np

# Kyte-Doolittle hydropathy (1982)
KD = {"A":  1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C":  2.5,
      "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I":  4.5,
      "L":  3.8, "K": -3.9, "M":  1.9, "F":  2.8, "P": -1.6,
      "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V":  4.2}

# side-chain volume (Å^3), Zamyatnin 1972 / Pommié 2004
SC_VOL = {"A":  88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5,
          "Q": 143.8, "E": 138.4, "G":  60.1, "H": 153.2, "I": 166.7,
          "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9, "P": 112.7,
          "S":  89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0}

POS = {"R", "K"}            # treat H as neutral at pH 7 (consistent with sim)
NEG = {"D", "E"}
CHARGED = POS | NEG
AROMATIC = {"F", "Y", "W"}

# ---------- per-chain primitives ------------------------------------------
def chain_f_pos(seq):     return sum(r in POS for r in seq) / len(seq)
def chain_f_neg(seq):     return sum(r in NEG for r in seq) / len(seq)
def chain_FCR(seq):       return chain_f_pos(seq) + chain_f_neg(seq)
def chain_NCPR(seq):      return chain_f_pos(seq) - chain_f_neg(seq)
def chain_f_aro(seq):     return sum(r in AROMATIC for r in seq) / len(seq)
def chain_f_sticker(seq): return sum(r != "G" for r in seq) / len(seq)
def chain_KD(seq):        return np.mean([KD[r] for r in seq])
def chain_volume(seq):    return np.mean([SC_VOL[r] for r in seq])


def chain_sticker_spread(seq):
    """Std-dev of sticker (non-G) positions, normalized by chain length.
    Returns 0 if 0 or 1 stickers."""
    positions = [i for i, r in enumerate(seq) if r != "G"]
    if len(positions) < 2:
        return 0.0
    return float(np.std(positions)) / len(seq)


def chain_kappa_xg(seq, blob=4):
    """Das-Pappu κ patterning with two classes: X (non-G) and G.

    σ(blob) = (f_X - f_G)^2 / (f_X + f_G)  per blob (always = 1 when only one class)
    κ = <σ_blob> / σ_max   where σ_max = (f_X - f_G)^2 / (f_X + f_G) of the whole.
    Returns 0 if the whole sequence has only one class (cannot pattern).
    """
    L = len(seq)
    is_x = np.array([r != "G" for r in seq], dtype=float)
    fx = is_x.mean()
    fg = 1 - fx
    if fx == 0 or fg == 0:
        return 0.0
    sigma_total = (fx - fg) ** 2  # /(fx+fg)=1
    if L < blob:
        return 0.0
    sigmas = []
    for i in range(L - blob + 1):
        win = is_x[i:i + blob]
        a = win.mean()
        b = 1 - a
        if (a + b) == 0:
            continue
        sigmas.append((a - b) ** 2 / (a + b))
    if not sigmas:
        return 0.0
    # FIX: was missing — must return κ = mean(σ_blob) / σ_total
    return float(np.mean(sigmas)) / sigma_total if sigma_total > 0 else 0.0


# ---------- system-level (mole-fraction weighted) -------------------------
PER_CHAIN_FUNCS = {
    "FCR":            chain_FCR,
    "NCPR_abs":       lambda s: abs(chain_NCPR(s)),
    "f_aromatic":     chain_f_aro,
    "f_sticker":      chain_f_sticker,
    "hydropathy":     chain_KD,
    "sc_volume":      chain_volume,
    "kappa_xg":       chain_kappa_xg,
    "sticker_spread": chain_sticker_spread,
}


def system_descriptors(sysname):
    """Return dict of mole-fraction-weighted system descriptors."""
    mix = SYSTEMS[sysname]
    N = total_peptides(sysname)
    out = {}
    for name, f in PER_CHAIN_FUNCS.items():
        out[name] = sum(count * f(seq) for seq, count in mix.items()) / N
    # also: mean # of stickers per chain (== f_sticker * chain length)
    out["n_stickers"] = sum(count * sum(r != "G" for r in seq)
                            for seq, count in mix.items()) / N
    return out


def all_systems_descriptor_table():
    import pandas as pd
    rows = []
    for s in SYSTEMS:
        d = system_descriptors(s)
        d["system"] = s
        rows.append(d)
    df = pd.DataFrame(rows).set_index("system")
    cols = ["FCR", "NCPR_abs", "f_aromatic", "f_sticker", "n_stickers",
            "hydropathy", "sc_volume", "kappa_xg", "sticker_spread"]
    return df[cols]



In [ ]:
"""Aggregate condensate observables per system from the existing CSVs and pickles.

Per system we report:
    water_per_peptide  (cluster_water CSV)
    waterbridges_per_res (waterbridges CSV / cluster_size)
    sasa_per_atom     (sasa_peratom_*.csv)
    tau_pw_eff        (peptide-water contact lifetime, from corr pickles)
    tau_in            (inner-core water survival lifetime via biexp fit of sp_*_new.csv)
    tau_out           (outer-layer water survival lifetime)
    tau_all           (overall water survival lifetime via biexp fit of sp_all_*_new.csv)
"""

import os
import pickle
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

DATA = "/Volumes/potoyan-lab/will/gxg-project/vysakh_analysis/data"
CORR = "/Volumes/potoyan-lab/will/gxg-project/vysakh_analysis/corr_results"

# system -> (comp_key, chain, pattern_key)
# comp_key matches the CSV filename root used by Vysakh
# For patterned systems pattern_key is used (e.g. "2_close").
COMP_FOR = {"3E": "equal", "3C": "charged", "3A": "aromatic",
            "3P": "polar", "3N": "nonpolar",
            "9E": "equal", "9C": "charged", "9A": "aromatic",
            "9P": "polar", "9N": "nonpolar"}
CHAIN_FOR = {**{k: 3 for k in ["3E", "3C", "3A", "3P", "3N"]},
             **{k: 9 for k in ["9E", "9C", "9A", "9P", "9N",
                               "X2", "X2S", "X3", "X3S"]}}
PATTERN_KEY = {"X2": "2_close", "X2S": "2_spread",
               "X3": "3_close", "X3S": "3_spread"}

ALL_SYSTEMS = ["3E", "3C", "3A", "3P", "3N",
               "9E", "9C", "9A", "9P", "9N",
               "X2", "X2S", "X3", "X3S"]


# ---------- fit helpers ---------------------------------------------------
def biexp(t, A, t1, t2):
    return A * np.exp(-t / t1) + (1 - A) * np.exp(-t / t2)


def fit_teff_biexp(y, dt=0.005):
    t = np.arange(len(y)) * dt
    try:
        p, _ = curve_fit(biexp, t, y, p0=(0.5, 0.01, 0.5),
                         bounds=([0, 1e-5, 1e-3], [1, 5, 50]),
                         maxfev=20000)
        A, t1, t2 = p
        return A * t1 + (1 - A) * t2
    except Exception:
        return np.nan


# ---------- per-system observable readers ---------------------------------
def _cluster_water_path(sys):
    chain = CHAIN_FOR[sys]
    if sys in PATTERN_KEY:
        return os.path.join(DATA, f"{PATTERN_KEY[sys]}_cluster_water_9res.csv")
    suf = "" if chain == 3 else "_9res"
    return os.path.join(DATA, f"{COMP_FOR[sys]}_cluster_water{suf}.csv")


def _waterbridges_path(sys):
    chain = CHAIN_FOR[sys]
    if sys in PATTERN_KEY:
        return os.path.join(DATA, f"{PATTERN_KEY[sys]}_waterbridges_9res.csv")
    return os.path.join(DATA, f"{COMP_FOR[sys]}_waterbridges_{chain}res.csv")


def _sp_path(sys, kind):
    """kind in {'inner_outer', 'all'}"""
    chain = CHAIN_FOR[sys]
    pkey = PATTERN_KEY.get(sys, COMP_FOR.get(sys))
    prefix = "sp" if kind == "inner_outer" else "sp_all"
    if sys in PATTERN_KEY:
        return os.path.join(DATA, f"{prefix}_9res_{pkey}_new.csv")
    return os.path.join(DATA, f"{prefix}_{chain}res_{pkey}_new.csv")


def _sasa_csv_for_chain(chain, is_pattern):
    if is_pattern:
        return os.path.join(DATA, "sasa_peratom_9res_spacers.csv")
    return os.path.join(DATA, f"sasa_peratom_{chain}res.csv")


def _sasa_col_for_sys(sys):
    if sys in PATTERN_KEY:
        return PATTERN_KEY[sys]
    return COMP_FOR[sys]



# ---------- correlation pickle access -------------------------------------
def _corr_teff(sys):
    """Return t_eff from peptide-water contact correlation pickle."""
    if CHAIN_FOR[sys] == 3:
        path = os.path.join(CORR, "3res_corr_results_2.pkl")
        key = COMP_FOR[sys]
    elif sys in PATTERN_KEY:
        path = os.path.join(CORR, "9res_corr_results_spacers.pkl")
        key = PATTERN_KEY[sys]
    else:
        path = os.path.join(CORR, "9res_corr_results_2.pkl")
        key = COMP_FOR[sys]
    if not os.path.exists(path):
        return np.nan
    with open(path, "rb") as fh:
        d = pickle.load(fh)
    return float(d[key]["t_eff"]) if key in d else np.nan


# ---------- main builder --------------------------------------------------
def all_systems_observables():
    rows = []
    for sys in ALL_SYSTEMS:
        row = {"system": sys, "chain": CHAIN_FOR[sys],
               "is_pattern": sys in PATTERN_KEY}
        # water per peptide and # waterbridges per residue (mean over trajectory)
        cw = _cluster_water_path(sys)
        if os.path.exists(cw):
            df = pd.read_csv(cw)
            row["water_per_peptide"] = float((df["water_mass"]
                                              / df["protein_mass"]).mean())
            n_res = df["cluster_size"].mean()
        else:
            row["water_per_peptide"] = np.nan
            n_res = np.nan
        wb = _waterbridges_path(sys)
        if os.path.exists(wb) and np.isfinite(n_res):
            wbdf = pd.read_csv(wb)
            col = "WaterBridgeCount" if "WaterBridgeCount" in wbdf else wbdf.columns[0]
            row["waterbridges_per_res"] = float(wbdf[col].mean() / n_res)
        else:
            row["waterbridges_per_res"] = np.nan
        # SASA per atom
        sasa_csv = _sasa_csv_for_chain(CHAIN_FOR[sys], sys in PATTERN_KEY)
        if os.path.exists(sasa_csv):
            sdf = pd.read_csv(sasa_csv)
            col = _sasa_col_for_sys(sys)
            if col in sdf.columns:
                row["sasa_per_atom"] = float(sdf[col].mean())
            else:
                row["sasa_per_atom"] = np.nan
        # tau peptide-water (from pickle, biexp fit already done)
        row["tau_pw_eff"] = _corr_teff(sys)
        # tau water survival, inner / outer / overall (refit biexp)
        sp_io = _sp_path(sys, "inner_outer")
        if os.path.exists(sp_io):
            d = pd.read_csv(sp_io)
            row["tau_in"]  = fit_teff_biexp(d["Inner"].values)
            row["tau_out"] = fit_teff_biexp(d["Outer"].values)
        sp_all = _sp_path(sys, "all")
        if os.path.exists(sp_all):
            d = pd.read_csv(sp_all)
            col = "SP_all" if "SP_all" in d.columns else d.columns[0]
            row["tau_all"] = fit_teff_biexp(d[col].values)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("system")
    return df

## Cluster_size

total_size_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/{i}/{j}_size.csv', 
                              index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = temp_df['prot-size'].values
        total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    #num_stickers = 1
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/close_spread/{i}_size.csv',
                          index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = temp_df['prot-size'].values / num_stickers
    total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)


total_size_df

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SYSTEMS
# ============================================================

systems = ['3E','3C','3A','3P','3N',
           '9E','9C','9C','9P','9N',
           'X2','X2S','X3','X3S']

# ============================================================
# COLLECT MEAN PERCOLATION VALUES
# ============================================================

perc_values = []

for label in systems:

    df = pd.read_csv(
        f'/Volumes/potoyan-lab/will/gxg-project/new_plots/data/net_metrics_{label}.csv'
    )

    perc_values.append(df['percolation_pc'].mean())

perc_values

In [ ]:
df_2 = all_systems_descriptor_table()
df_2.head()

In [ ]:
df = all_systems_observables()
df.head()

In [ ]:

df['cluster_size'] = total_size_df.mean()


In [ ]:

df_2['chain'] = df['chain']


In [ ]:

df['percolation'] = perc_values


In [ ]:
## Add Protein t

df['protein t'] = all_tau_protein


In [ ]:
df_2.to_csv('Figures/descriptor.csv')
df.to_csv('Figures/observables.csv')

In [ ]:
pd.read_csv('Figures/observables.csv')